<a href="https://colab.research.google.com/github/al12515037-tech/proyectos-ia/blob/main/M2_T2_HandsOn_Diseno_Agentes_Conversacionales_OscarCabrales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: DISEÑO DE AGENTES CONVERSACIONALES**

Una vez vista la masterclass ***Diseño de Agentes Conversacionales***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.


De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/18wkM2LSfY39VZ2GBdcPCY2Qw0EzewJsx?usp=drive_link).

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**


#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# ============================================================
# CELDA 1 - CARGAR CREDENCIALES DESDE COLAB SECRETS
# ============================================================

from google.colab import userdata

# Recuperar credenciales de forma segura
WHATSAPP_ACCESS_TOKEN = userdata.get("WHATSAPP_ACCESS_TOKEN")
VERIFY_TOKEN = userdata.get("VERIFY_TOKEN")
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")

# Verificación sin mostrar los valores secretos
print("🔐 Verificación de Colab Secrets")
print("--------------------------------")

print(
    "WHATSAPP_ACCESS_TOKEN:",
    "✅ CARGADO" if WHATSAPP_ACCESS_TOKEN else "❌ NO CARGADO"
)

print(
    "VERIFY_TOKEN:",
    "✅ CARGADO" if VERIFY_TOKEN else "❌ NO CARGADO"
)

print(
    "NGROK_AUTHTOKEN:",
    "✅ CARGADO" if NGROK_AUTHTOKEN else "❌ NO CARGADO"
)

if WHATSAPP_ACCESS_TOKEN and VERIFY_TOKEN and NGROK_AUTHTOKEN:
    print("\n✅ Todas las credenciales fueron cargadas correctamente")
else:
    print("\n⚠️ Falta una o más credenciales en Colab Secrets")


🔐 Verificación de Colab Secrets
--------------------------------
WHATSAPP_ACCESS_TOKEN: ✅ CARGADO
VERIFY_TOKEN: ✅ CARGADO
NGROK_AUTHTOKEN: ✅ CARGADO

✅ Todas las credenciales fueron cargadas correctamente


#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [ ]:
# ============================================================
# CELDA 2 - INSTALAR OLLAMA EN GOOGLE COLAB
# ============================================================

!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

print("\n✅ Instalación de Ollama finalizada.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

✅ Instalación de Ollama finalizada.


In [ ]:
# ============================================================
# CELDA 3 - INICIAR SERVIDOR OLLAMA
# ============================================================

import subprocess
import time
import requests

# Iniciar Ollama en segundo plano
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("⏳ Iniciando servidor Ollama...")
time.sleep(5)

# Comprobar que la API está disponible
try:
    response = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)

    if response.status_code == 200:
        print("✅ Servidor Ollama activo.")
        print("🌐 API disponible en http://127.0.0.1:11434")
    else:
        print("⚠️ Ollama respondió con código:", response.status_code)

except Exception as e:
    print("❌ No fue posible conectar con Ollama:", e)


⏳ Iniciando servidor Ollama...
✅ Servidor Ollama activo.
🌐 API disponible en http://127.0.0.1:11434


#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [ ]:
# ============================================================
# CELDA 4 - INSTALAR DEPENDENCIAS DEL AGENTE
# ============================================================

!pip install -q \
    fastapi \
    uvicorn \
    requests \
    pyngrok \
    ollama \
    pyyaml

print("✅ Dependencias principales instaladas correctamente.")


✅ Dependencias principales instaladas correctamente.


In [ ]:
# ============================================================
# CELDA 5 - INSTALAR Y LEVANTAR LLAMA STACK
# ============================================================

import subprocess
import time
import yaml
import requests

MODELO_LLAMA = "llama3.1:8b"

# ------------------------------------------------------------
# 1. VERIFICAR / DESCARGAR LLAMA 3.1:8B EN OLLAMA
# ------------------------------------------------------------

print("🔎 Verificando modelo Llama 3.1 en Ollama...")

respuesta = requests.get(
    "http://127.0.0.1:11434/api/tags",
    timeout=10
)

respuesta.raise_for_status()

modelos = [
    modelo.get("name", "")
    for modelo in respuesta.json().get("models", [])
]

if MODELO_LLAMA not in modelos:
    print("📥 Descargando llama3.1:8b...")
    print("⏳ Puede tardar varios minutos.\n")

    subprocess.run(
        ["ollama", "pull", MODELO_LLAMA],
        check=True
    )

    print("\n✅ Modelo llama3.1:8b descargado.")
else:
    print("✅ Modelo llama3.1:8b ya está disponible.")


# ------------------------------------------------------------
# 2. INSTALAR LLAMA STACK: SERVIDOR + CLIENTE
# ------------------------------------------------------------

print("\n📦 Instalando Llama Stack...")

subprocess.run(
    [
        "pip", "install", "-q",
        "llama-stack",
        "llama-stack-client"
    ],
    check=True
)

print("✅ Llama Stack instalado.")


# ------------------------------------------------------------
# 3. GENERAR CONFIGURACIÓN PARA OLLAMA
# ------------------------------------------------------------

config = {
    "version": "2",
    "image_name": "ollama",
    "apis": ["inference"],
    "providers": {
        "inference": [
            {
                "provider_id": "ollama",
                "provider_type": "remote::ollama",
                "config": {
                    "url": "http://127.0.0.1:11434"
                }
            }
        ]
    },
    "registered_resources": {
        "models": [
            {
                "model_id": MODELO_LLAMA,
                "provider_id": "ollama",
                "provider_model_id": MODELO_LLAMA,
                "model_type": "llm"
            }
        ]
    }
}

mi_config_path = "/content/mi_config_llama_stack.yaml"

with open(mi_config_path, "w") as f:
    yaml.dump(
        config,
        f,
        default_flow_style=False,
        sort_keys=False
    )

print("✅ Configuración creada:")
print(mi_config_path)


# ------------------------------------------------------------
# 4. LEVANTAR SERVIDOR LLAMA STACK
# ------------------------------------------------------------

print("\n🚀 Iniciando Llama Stack...")

proceso_llama_stack = subprocess.Popen(
    [
        "llama", "stack", "run",
        mi_config_path,
        "--port", "8321"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Llama Stack tarda más en iniciar que Ollama
time.sleep(20)

print(
    "🔎 Proceso Llama Stack activo:",
    proceso_llama_stack.poll() is None
)

if proceso_llama_stack.poll() is not None:
    print("\n❌ Llama Stack terminó inesperadamente.")
    print(proceso_llama_stack.stdout.read())

else:
    from llama_stack_client import LlamaStackClient

    client = LlamaStackClient(
        base_url="http://localhost:8321"
    )

    print("✅ Llama Stack listo.")
    print("🌐 Servidor: http://localhost:8321")
    print("🤖 Modelo registrado:", MODELO_LLAMA)


🔎 Verificando modelo Llama 3.1 en Ollama...
📥 Descargando llama3.1:8b...
⏳ Puede tardar varios minutos.


✅ Modelo llama3.1:8b descargado.

📦 Instalando Llama Stack...
✅ Llama Stack instalado.
✅ Configuración creada:
/content/mi_config_llama_stack.yaml

🚀 Iniciando Llama Stack...
🔎 Proceso Llama Stack activo: False

❌ Llama Stack terminó inesperadamente.
Traceback (most recent call last):
  File "/usr/local/bin/llama", line 5, in <module>
    from llama_stack.cli.llama import main
  File "/usr/local/lib/python3.13/dist-packages/llama_stack/cli/llama.py", line 14, in <module>
    from .stack import StackParser  # type: ignore[attr-defined]
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/llama_stack/cli/stack/__init__.py", line 7, in <module>
    from .stack import StackParser  # noqa
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/llama_stack/cli/stack/stack.py", line 18, in <module>
    from .run import StackRun
  Fi

In [ ]:
# ============================================================
# CELDA 5A - CORREGIR COMPATIBILIDAD LLAMA STACK / MCP
# ============================================================

import importlib.metadata as metadata

print("🔎 Versiones instaladas antes de la corrección:")
print("llama-stack:", metadata.version("llama-stack"))
print("mcp:", metadata.version("mcp"))

# Instalar una versión de MCP compatible con Llama Stack
!pip install -q --upgrade "mcp<1.10"

print("\n🔎 Versión MCP después de la corrección:")
print("mcp:", metadata.version("mcp"))

# Comprobar específicamente el símbolo que necesita Llama Stack
try:
    from mcp import McpError
    print("✅ McpError disponible.")
    print("✅ Compatibilidad MCP / Llama Stack corregida.")
except ImportError as e:
    print("❌ McpError todavía no está disponible:")
    print(e)

🔎 Versiones instaladas antes de la corrección:
llama-stack: 0.7.3
mcp: 2.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-stack 0.7.3 requires mcp>=1.23.0, but you have mcp 1.9.4 which is incompatible.

🔎 Versión MCP después de la corrección:
mcp: 1.9.4
✅ McpError disponible.
✅ Compatibilidad MCP / Llama Stack corregida.


In [ ]:
# ============================================================
# CELDA 5B - AJUSTAR MCP A UNA VERSIÓN COMPATIBLE
# ============================================================

!pip install -q --upgrade "mcp>=1.23.0,<2.0.0"

import importlib.metadata as metadata

print("🔎 Versiones después del ajuste:")
print("llama-stack:", metadata.version("llama-stack"))
print("mcp:", metadata.version("mcp"))

try:
    from mcp import McpError
    print("✅ McpError disponible.")
    print("✅ Requisito de Llama Stack satisfecho.")
except ImportError as e:
    print("❌ Persisten problemas de compatibilidad:")
    print(e)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 7.7 MB/s eta 0:00:00
🔎 Versiones después del ajuste:
llama-stack: 0.7.3
mcp: 1.30.0
✅ McpError disponible.
✅ Requisito de Llama Stack satisfecho.


In [ ]:
# ============================================================
# CELDA 5C - LEVANTAR LLAMA STACK DESPUÉS DE CORREGIR MCP
# ============================================================

import subprocess
import time

mi_config_path = "/content/mi_config_llama_stack.yaml"

print("🚀 Iniciando Llama Stack...")
print("📄 Configuración:", mi_config_path)

proceso_llama_stack = subprocess.Popen(
    [
        "llama", "stack", "run",
        mi_config_path,
        "--port", "8321"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("⏳ Esperando que Llama Stack termine de iniciar...")
time.sleep(20)

if proceso_llama_stack.poll() is None:

    print("✅ Proceso Llama Stack activo.")

    try:
        from llama_stack_client import LlamaStackClient

        cliente = LlamaStackClient(
            base_url="http://localhost:8321"
        )

        print("✅ Cliente Llama Stack creado correctamente.")
        print("🌐 Servidor: http://localhost:8321")
        print("🤖 Modelo esperado: llama3.1:8b")

    except Exception as e:
        print("⚠️ El servidor está activo, pero ocurrió un problema al crear el cliente:")
        print(e)

else:
    print("❌ Llama Stack volvió a detenerse.")
    print("\n===== LOG DEL SERVIDOR =====")
    print(proceso_llama_stack.stdout.read())

🚀 Iniciando Llama Stack...
📄 Configuración: /content/mi_config_llama_stack.yaml
⏳ Esperando que Llama Stack termine de iniciar...
✅ Proceso Llama Stack activo.
✅ Cliente Llama Stack creado correctamente.
🌐 Servidor: http://localhost:8321
🤖 Modelo esperado: llama3.1:8b


In [ ]:
# ============================================================
# CELDA 5D - PRUEBA DE INFERENCIA LLAMA STACK → LLAMA 3.1
# ============================================================

from llama_stack_client import LlamaStackClient

# Crear/reutilizar cliente de Llama Stack
cliente = LlamaStackClient(
    base_url="http://localhost:8321"
)

MODELO_LLAMA = "llama3.1:8b"

print("🧪 PRUEBA DE COMUNICACIÓN")
print("Llama Stack → Ollama → Llama 3.1")
print("----------------------------------------")

try:
    respuesta = cliente.inference.chat_completion(
        model_id=MODELO_LLAMA,
        messages=[
            {
                "role": "system",
                "content": (
                    "Eres un asistente de inteligencia artificial. "
                    "Responde de forma clara, breve y en español."
                )
            },
            {
                "role": "user",
                "content": (
                    "Responde en una sola oración: "
                    "¿qué función cumple la memoria en un agente conversacional?"
                )
            }
        ]
    )

    print("✅ Solicitud procesada por Llama Stack.")
    print("\n🤖 RESPUESTA DE LLAMA 3.1:")
    print("----------------------------------------")

    # Extraer el texto de la respuesta
    try:
        texto = respuesta.completion_message.content
    except Exception:
        texto = str(respuesta)

    print(texto)

    print("\n----------------------------------------")
    print("✅ PRUEBA DE INFERENCIA FINALIZADA.")

except Exception as error:
    print("❌ Error durante la prueba de inferencia:")
    print(type(error).__name__)
    print(error)

🧪 PRUEBA DE COMUNICACIÓN
Llama Stack → Ollama → Llama 3.1
----------------------------------------
❌ Error durante la prueba de inferencia:
AttributeError
'LlamaStackClient' object has no attribute 'inference'


In [ ]:
# ============================================================
# CELDA 5D-1 - INSPECCIONAR API DISPONIBLE EN LLAMA STACK
# ============================================================

from llama_stack_client import LlamaStackClient
import importlib.metadata as metadata

cliente = LlamaStackClient(
    base_url="http://localhost:8321"
)

print("DIAGNOSTICO DEL CLIENTE LLAMA STACK")
print("----------------------------------------")
print("llama-stack:", metadata.version("llama-stack"))
print("llama-stack-client:", metadata.version("llama-stack-client"))

print("\nMetodos y recursos disponibles en 'cliente':")

metodos = [
    nombre
    for nombre in dir(cliente)
    if not nombre.startswith("_")
]

for nombre in metodos:
    print(nombre)

print("\nComprobaciones:")
print("Tiene 'inference':", hasattr(cliente, "inference"))
print("Tiene 'chat':", hasattr(cliente, "chat"))
print("Tiene 'responses':", hasattr(cliente, "responses"))
print("Tiene 'models':", hasattr(cliente, "models"))

DIAGNOSTICO DEL CLIENTE LLAMA STACK
----------------------------------------
llama-stack: 0.7.3
llama-stack-client: 0.7.5

Metodos y recursos disponibles en 'cliente':
alpha
api_key
auth_headers
base_url
batches
beta
chat
close
completions
conversations
copy
custom_auth
default_headers
default_query
delete
embeddings
files
get
get_api_list
inspect
is_closed
max_retries
models
moderations
patch
platform_headers
post
prompts
providers
put
qs
request
responses
routes
safety
scoring
scoring_functions
shields
timeout
user_agent
vector_io
vector_stores
with_options
with_raw_response
with_streaming_response

Comprobaciones:
Tiene 'inference': False
Tiene 'chat': True
Tiene 'responses': True
Tiene 'models': True


In [ ]:
# ============================================================
# CELDA 5D-2 - PRUEBA DE INFERENCIA MEDIANTE CHAT
# ============================================================

MODELO_LLAMA = "llama3.1:8b"

print("PRUEBA DE COMUNICACION")
print("Llama Stack -> Ollama -> Llama 3.1")
print("----------------------------------------")

try:
    respuesta = cliente.chat.completions.create(
        model=MODELO_LLAMA,
        messages=[
            {
                "role": "system",
                "content": (
                    "Eres un asistente de inteligencia artificial. "
                    "Responde de forma clara, breve y en español."
                )
            },
            {
                "role": "user",
                "content": (
                    "Responde en una sola oracion: "
                    "¿que funcion cumple la memoria en un agente conversacional?"
                )
            }
        ]
    )

    print("Solicitud procesada por Llama Stack.")
    print("\nRESPUESTA DE LLAMA 3.1:")
    print("----------------------------------------")

    texto = respuesta.choices[0].message.content
    print(texto)

    print("----------------------------------------")
    print("PRUEBA DE INFERENCIA FINALIZADA CORRECTAMENTE.")

except Exception as error:
    print("ERROR DURANTE LA PRUEBA DE INFERENCIA")
    print("Tipo:", type(error).__name__)
    print("Detalle:", error)

PRUEBA DE COMUNICACION
Llama Stack -> Ollama -> Llama 3.1
----------------------------------------
ERROR DURANTE LA PRUEBA DE INFERENCIA
Tipo: NotFoundError
Detalle: Error code: 404 - {'error': {'message': "Model 'llama3.1:8b' not found. Use 'client.models.list()' to list available Models."}}


In [ ]:
# ============================================================
# CELDA 5D-3 - LISTAR MODELOS REGISTRADOS EN LLAMA STACK
# ============================================================

print("MODELOS REGISTRADOS EN LLAMA STACK")
print("----------------------------------------")

try:
    modelos = cliente.models.list()

    print(modelos)

    print("\n----------------------------------------")
    print("Consulta de modelos finalizada.")

except Exception as error:
    print("ERROR AL CONSULTAR LOS MODELOS")
    print("Tipo:", type(error).__name__)
    print("Detalle:", error)

MODELOS REGISTRADOS EN LLAMA STACK
----------------------------------------
[Model(id='ollama/llama3.1:8b', created=1790560701, owned_by='llama_stack', custom_metadata={'model_type': 'llm', 'provider_id': 'ollama', 'provider_resource_id': 'llama3.1:8b'}, object='model')]

----------------------------------------
Consulta de modelos finalizada.


In [ ]:
# ============================================================
# CELDA 5D-4 - PRUEBA DEFINITIVA DE INFERENCIA
# ============================================================

MODELO_LLAMA_STACK = "ollama/llama3.1:8b"

print("PRUEBA DE COMUNICACION")
print("Llama Stack -> Ollama -> Llama 3.1")
print("----------------------------------------")
print("Modelo:", MODELO_LLAMA_STACK)

try:
    respuesta = cliente.chat.completions.create(
        model=MODELO_LLAMA_STACK,
        messages=[
            {
                "role": "system",
                "content": (
                    "Eres un asistente de inteligencia artificial. "
                    "Responde de forma clara, breve y en español."
                )
            },
            {
                "role": "user",
                "content": (
                    "Responde en una sola oracion: "
                    "¿que funcion cumple la memoria en un agente conversacional?"
                )
            }
        ]
    )

    texto = respuesta.choices[0].message.content

    print("\nRESPUESTA DE LLAMA 3.1:")
    print("----------------------------------------")
    print(texto)
    print("----------------------------------------")

    if texto and texto.strip():
        print("PRUEBA DE INFERENCIA FINALIZADA CORRECTAMENTE.")
    else:
        print("La solicitud fue procesada, pero no se recibio texto.")

except Exception as error:
    print("ERROR DURANTE LA PRUEBA DE INFERENCIA")
    print("Tipo:", type(error).__name__)
    print("Detalle:", error)

PRUEBA DE COMUNICACION
Llama Stack -> Ollama -> Llama 3.1
----------------------------------------
Modelo: ollama/llama3.1:8b
ERROR DURANTE LA PRUEBA DE INFERENCIA
Tipo: APITimeoutError
Detalle: Request timed out.


In [ ]:
# ============================================================
# CELDA 5D-5 - PRUEBA DE INFERENCIA CON TIMEOUT AMPLIADO
# ============================================================

from llama_stack_client import LlamaStackClient
import time

MODELO_LLAMA_STACK = "ollama/llama3.1:8b"

cliente_prueba = LlamaStackClient(
    base_url="http://localhost:8321",
    timeout=300.0
)

print("PRUEBA DE INFERENCIA CON TIMEOUT AMPLIADO")
print("----------------------------------------")
print("Modelo:", MODELO_LLAMA_STACK)
print("Tiempo maximo de espera: 300 segundos")
print("Procesando solicitud...")

inicio = time.time()

try:
    respuesta = cliente_prueba.chat.completions.create(
        model=MODELO_LLAMA_STACK,
        messages=[
            {
                "role": "system",
                "content": (
                    "Eres un asistente de inteligencia artificial. "
                    "Responde brevemente y en español."
                )
            },
            {
                "role": "user",
                "content": (
                    "¿Que funcion cumple la memoria "
                    "en un agente conversacional?"
                )
            }
        ],
        max_tokens=80
    )

    tiempo = time.time() - inicio
    texto = respuesta.choices[0].message.content

    print("\nRESPUESTA DE LLAMA 3.1:")
    print("----------------------------------------")
    print(texto)
    print("----------------------------------------")
    print(f"Tiempo de respuesta: {tiempo:.1f} segundos")

    if texto and texto.strip():
        print("PRUEBA DE INFERENCIA FINALIZADA CORRECTAMENTE.")
    else:
        print("La solicitud termino, pero no se recibio texto.")

except Exception as error:
    tiempo = time.time() - inicio

    print("\nERROR DURANTE LA PRUEBA")
    print("Tipo:", type(error).__name__)
    print("Detalle:", error)
    print(f"Tiempo transcurrido: {tiempo:.1f} segundos")

PRUEBA DE INFERENCIA CON TIMEOUT AMPLIADO
----------------------------------------
Modelo: ollama/llama3.1:8b
Tiempo maximo de espera: 300 segundos
Procesando solicitud...

RESPUESTA DE LLAMA 3.1:
----------------------------------------
La memoria en un agente conversacional almacena conversaciones y contextos anteriores para proporcionar continuidad y precisión en la respuesta. Este componente le permite recordar detalles de la conversación y mantener una conversación coherente.
----------------------------------------
Tiempo de respuesta: 105.1 segundos
PRUEBA DE INFERENCIA FINALIZADA CORRECTAMENTE.


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [ ]:
# ============================================================
# CONFIGURAR EL SERVIDOR DEL WEBHOOK
# ============================================================

from fastapi import FastAPI, Request, BackgroundTasks
from fastapi.responses import PlainTextResponse, JSONResponse
import requests

app = FastAPI()

# Estructuras para almacenar informacion durante la sesion
mensajes_recibidos = []

# Estado conversacional asociado al numero de WhatsApp
estado_usuarios = {}

print("Servidor FastAPI configurado correctamente")
print("Estructura de mensajes creada")
print("Estructura de estado conversacional creada")


Servidor FastAPI configurado correctamente
Estructura de mensajes creada
Estructura de estado conversacional creada


In [ ]:
# ============================================================
# DEFINIR FUNCION DE NORMALIZACION DE NUMERO
# ============================================================

import re

def normalizar_numero(numero):
    """
    Normaliza numeros telefonicos para utilizarlos
    como identificador unico del usuario en WhatsApp.

    - Elimina espacios, guiones, parentesis y el simbolo +.
    - Convierte numeros mexicanos de 10 digitos al formato 52XXXXXXXXXX.
    - Corrige el formato antiguo 521XXXXXXXXXX a 52XXXXXXXXXX.
    """

    # Conservar unicamente digitos
    numero = re.sub(r"\D", "", str(numero))

    # Formato antiguo de Mexico: 521 + 10 digitos
    if numero.startswith("521") and len(numero) == 13:
        numero = "52" + numero[3:]

    # Numero mexicano nacional de 10 digitos
    elif len(numero) == 10:
        numero = "52" + numero

    # Validacion basica
    if not numero.isdigit() or len(numero) < 10 or len(numero) > 15:
        raise ValueError(
            f"Numero telefonico no valido para WhatsApp: {numero}"
        )

    return numero


# Prueba controlada
numero_prueba = "933 119 6350"
numero_normalizado = normalizar_numero(numero_prueba)

print("Funcion normalizar_numero() creada correctamente")
print("Numero de prueba:", numero_prueba)
print("Numero normalizado:", numero_normalizado)

assert numero_normalizado == "529331196350"

print("Prueba de normalizacion superada")


Funcion normalizar_numero() creada correctamente
Numero de prueba: 933 119 6350
Numero normalizado: 529331196350
Prueba de normalizacion superada


#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que guarda, por cada número de teléfono, el historial reciente de mensajes — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación. Esta es una implementación manual y simplificada del concepto; la lectura del Tema menciona **Llama Stack** como el framework que estandariza esta pieza (junto con herramientas y seguridad) para no reconstruirla en cada proyecto. **Nota:** el patrón exacto de integración de Llama Stack debe confirmarse contra su documentación vigente antes de grabar o dar la sesión en vivo.

In [ ]:
# ============================================================
# MEMORIA DE CONVERSACION POR NUMERO DE TELEFONO
# ============================================================

# Diccionario principal de memoria
# Cada numero de WhatsApp tendra su propio historial.
memoria_conversaciones = {}

# Numero maximo de mensajes recientes que se conservaran
MAX_MENSAJES_HISTORIAL = 10


def obtener_historial(numero):
    """
    Obtiene el historial asociado a un numero de telefono.
    Si el usuario aun no existe, crea una conversacion vacia.
    """

    numero = normalizar_numero(numero)

    if numero not in memoria_conversaciones:
        memoria_conversaciones[numero] = []

    return memoria_conversaciones[numero]


def guardar_mensaje(numero, rol, contenido):
    """
    Guarda un mensaje dentro del historial del usuario.

    rol:
        user      -> mensaje enviado por el usuario
        assistant -> respuesta generada por el asistente
    """

    numero = normalizar_numero(numero)

    historial = obtener_historial(numero)

    historial.append(
        {
            "role": rol,
            "content": contenido
        }
    )

    # Mantener solamente los mensajes mas recientes
    if len(historial) > MAX_MENSAJES_HISTORIAL:
        memoria_conversaciones[numero] = historial[
            -MAX_MENSAJES_HISTORIAL:
        ]


# ------------------------------------------------------------
# PRUEBA CONTROLADA
# ------------------------------------------------------------

numero_prueba_memoria = "933 119 6350"

guardar_mensaje(
    numero_prueba_memoria,
    "user",
    "Quiero una cita dental."
)

historial_prueba = obtener_historial(
    numero_prueba_memoria
)

print("Memoria conversacional creada correctamente")
print("Numero:", normalizar_numero(numero_prueba_memoria))
print("Mensajes almacenados:", len(historial_prueba))
print("Historial:", historial_prueba)

assert len(historial_prueba) == 1
assert historial_prueba[0]["role"] == "user"

print("Prueba de memoria superada")


Memoria conversacional creada correctamente
Numero: 529331196350
Mensajes almacenados: 1
Historial: [{'role': 'user', 'content': 'Quiero una cita dental.'}]
Prueba de memoria superada


#### **GENERAR LA RESPUESTA CON HISTORIAL Y UNA REGLA FIJA**

Se define la lógica que arma cada respuesta: si el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento (una decisión que conviene fijar por regla, no dejarla al modelo); en cualquier otro caso, se reenvía todo el historial reciente a Llama, para que pueda resolver referencias a mensajes anteriores.

In [ ]:
# ============================================================
# GENERACION DE RESPUESTA CON MEMORIA Y REGLA FIJA
# ============================================================

MODELO_LLAMA_STACK = "ollama/llama3.1:8b"


def requiere_atencion_humana(texto):
    """
    Determina si el usuario solicita explicitamente
    hablar con una persona.
    """

    texto = texto.lower().strip()

    frases_escalamiento = [
        "hablar con una persona",
        "hablar con un humano",
        "hablar con un asesor",
        "hablar con un agente",
        "quiero un asesor",
        "quiero hablar con alguien"
    ]

    return any(
        frase in texto
        for frase in frases_escalamiento
    )


def generar_respuesta_con_memoria(numero, texto_usuario):
    """
    Genera una respuesta considerando el historial
    reciente asociado al numero de WhatsApp.

    Si el usuario solicita atencion humana,
    se aplica una regla fija y no se consulta al modelo.
    """

    numero = normalizar_numero(numero)
    texto_usuario = texto_usuario.strip()

    if not texto_usuario:
        raise ValueError("El mensaje del usuario no puede estar vacio.")

    # --------------------------------------------------------
    # REGLA FIJA DE ESCALAMIENTO
    # --------------------------------------------------------

    if requiere_atencion_humana(texto_usuario):

        respuesta_fija = (
            "Entiendo. Voy a canalizar tu solicitud "
            "para que puedas recibir atencion de una persona."
        )

        guardar_mensaje(
            numero,
            "user",
            texto_usuario
        )

        guardar_mensaje(
            numero,
            "assistant",
            respuesta_fija
        )

        return respuesta_fija

    # --------------------------------------------------------
    # RECUPERAR HISTORIAL
    # --------------------------------------------------------

    historial = obtener_historial(numero)

    mensajes_modelo = [
        {
            "role": "system",
            "content": (
                "Eres un asistente conversacional. "
                "Responde de forma clara, breve y en español. "
                "Utiliza el historial de la conversacion para "
                "comprender referencias a mensajes anteriores."
            )
        }
    ]

    # Incorporar historial previo
    mensajes_modelo.extend(historial)

    # Incorporar mensaje actual
    mensajes_modelo.append(
        {
            "role": "user",
            "content": texto_usuario
        }
    )

    # --------------------------------------------------------
    # CONSULTAR LLAMA STACK
    # --------------------------------------------------------

    respuesta = cliente_prueba.chat.completions.create(
        model=MODELO_LLAMA_STACK,
        messages=mensajes_modelo,
        max_tokens=120
    )

    texto_respuesta = (
        respuesta.choices[0].message.content.strip()
    )

    # --------------------------------------------------------
    # ACTUALIZAR MEMORIA
    # --------------------------------------------------------

    guardar_mensaje(
        numero,
        "user",
        texto_usuario
    )

    guardar_mensaje(
        numero,
        "assistant",
        texto_respuesta
    )

    return texto_respuesta


print("Funcion generar_respuesta_con_memoria() creada correctamente")
print("Regla fija de escalamiento configurada")
print("Historial conversacional conectado con Llama Stack")


Funcion generar_respuesta_con_memoria() creada correctamente
Regla fija de escalamiento configurada
Historial conversacional conectado con Llama Stack


In [ ]:
# ============================================================
# PRUEBA CONTROLADA DE MEMORIA CONVERSACIONAL
# ============================================================

numero_prueba = "933 119 6350"
numero_normalizado = normalizar_numero(numero_prueba)

# Limpiar solamente el historial del usuario de prueba
memoria_conversaciones[numero_normalizado] = []

print("PRUEBA DE MEMORIA CONVERSACIONAL")
print("----------------------------------------")
print("Numero:", numero_normalizado)


# ------------------------------------------------------------
# PRIMER MENSAJE
# ------------------------------------------------------------

mensaje_1 = "Quiero una cita dental."

print("\nPRIMER MENSAJE")
print("Usuario:", mensaje_1)
print("Procesando...")

respuesta_1 = generar_respuesta_con_memoria(
    numero_prueba,
    mensaje_1
)

print("Asistente:", respuesta_1)


# ------------------------------------------------------------
# SEGUNDO MENSAJE
# ------------------------------------------------------------

mensaje_2 = "¿A que hora abren los sabados?"

print("\nSEGUNDO MENSAJE")
print("Usuario:", mensaje_2)
print("Procesando...")

respuesta_2 = generar_respuesta_con_memoria(
    numero_prueba,
    mensaje_2
)

print("Asistente:", respuesta_2)


# ------------------------------------------------------------
# MOSTRAR MEMORIA FINAL
# ------------------------------------------------------------

historial_final = obtener_historial(numero_prueba)

print("\nHISTORIAL ALMACENADO")
print("----------------------------------------")

for indice, mensaje in enumerate(historial_final, start=1):
    print(
        indice,
        "-",
        mensaje["role"],
        ":",
        mensaje["content"]
    )

print("----------------------------------------")
print("Mensajes almacenados:", len(historial_final))

PRUEBA DE MEMORIA CONVERSACIONAL
----------------------------------------
Numero: 529331196350

PRIMER MENSAJE
Usuario: Quiero una cita dental.
Procesando...
Asistente: Podría ser 6 a 8 pm, dependiendo de la disponibilidad de tu dentista. ¿Quieres que te ayude a programar una cita en tu agenda o necesitas más información sobre cómo reservar?

SEGUNDO MENSAJE
Usuario: ¿A que hora abren los sabados?
Procesando...
Asistente: Por lo general, las consultas de odontología pueden tener un horario específico para los sábados, generalmente al mediodía o por la tarde. Revisaría con tu dentista qué horarios están disponibles para citas de sabado. En la mayoría de las clínicas dentales abren de 9:00 a 13:00 o 14:00 a 18:00 horas en sábados.

HISTORIAL ALMACENADO
----------------------------------------
1 - user : Quiero una cita dental.
2 - assistant : Podría ser 6 a 8 pm, dependiendo de la disponibilidad de tu dentista. ¿Quieres que te ayude a programar una cita en tu agenda o necesitas más infor

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [ ]:
# ============================================================
# ENDPOINT DE VERIFICACION DE CREDENCIALES
# ============================================================

from fastapi import Request
from fastapi.responses import PlainTextResponse


@app.get("/webhook")
async def verificar_webhook(request: Request):
    """
    Verifica el webhook solicitado por Meta.

    Meta envia:
    - hub.mode
    - hub.verify_token
    - hub.challenge

    Si el token recibido coincide con VERIFY_TOKEN,
    se devuelve el challenge con codigo HTTP 200.
    """

    mode = request.query_params.get("hub.mode")
    token = request.query_params.get("hub.verify_token")
    challenge = request.query_params.get("hub.challenge")

    print("Solicitud de verificacion recibida")
    print("Modo:", mode)

    if mode == "subscribe" and token == VERIFY_TOKEN:

        print("Webhook verificado correctamente")

        return PlainTextResponse(
            content=challenge or "",
            status_code=200
        )

    print("Verificacion rechazada")

    return PlainTextResponse(
        content="Token de verificacion incorrecto",
        status_code=403
    )


print("Endpoint GET /webhook creado correctamente")


Endpoint GET /webhook creado correctamente


In [ ]:
# ============================================================
# ENDPOINT DE RECEPCION DE MENSAJES
# ============================================================

from fastapi import Request, BackgroundTasks
from fastapi.responses import JSONResponse


@app.post("/webhook")
async def recibir_mensaje(
    request: Request,
    background_tasks: BackgroundTasks
):
    """
    Recibe las notificaciones enviadas por WhatsApp Cloud API.

    Si el evento contiene un mensaje de texto:
    1. Guarda el evento recibido.
    2. Obtiene el numero del remitente.
    3. Obtiene el contenido del mensaje.
    4. Delega el procesamiento a una tarea en segundo plano.
    5. Responde inmediatamente HTTP 200 a Meta.
    """

    try:
        datos = await request.json()

        # Guardar el evento para inspeccion y depuracion
        mensajes_recibidos.append(datos)

        # Extraer la estructura principal enviada por Meta
        entry = datos.get("entry", [])

        if not entry:
            return JSONResponse(
                content={"status": "ok"},
                status_code=200
            )

        changes = entry[0].get("changes", [])

        if not changes:
            return JSONResponse(
                content={"status": "ok"},
                status_code=200
            )

        value = changes[0].get("value", {})
        messages = value.get("messages", [])

        # Algunos eventos de Meta no contienen mensajes
        if not messages:
            return JSONResponse(
                content={"status": "ok"},
                status_code=200
            )

        mensaje = messages[0]

        numero = mensaje.get("from", "")
        tipo = mensaje.get("type", "")

        # Procesar solamente mensajes de texto
        if tipo == "text":

            texto = (
                mensaje
                .get("text", {})
                .get("body", "")
                .strip()
            )

            numero = normalizar_numero(numero)

            print("Mensaje recibido")
            print("Numero:", numero)
            print("Texto:", texto)

            if texto:
                background_tasks.add_task(
                    procesar_mensaje_whatsapp,
                    numero,
                    texto
                )

        return JSONResponse(
            content={"status": "ok"},
            status_code=200
        )

    except Exception as error:

        print("Error al procesar el webhook")
        print("Tipo:", type(error).__name__)
        print("Detalle:", error)

        return JSONResponse(
            content={"status": "error"},
            status_code=200
        )


print("Endpoint POST /webhook creado correctamente")


Endpoint POST /webhook creado correctamente


In [ ]:
# ============================================================
# CARGAR PHONE_NUMBER_ID DESDE COLAB SECRETS
# ============================================================

from google.colab import userdata

PHONE_NUMBER_ID = userdata.get("PHONE_NUMBER_ID")

if PHONE_NUMBER_ID:
    print("PHONE_NUMBER_ID cargado correctamente")
else:
    raise RuntimeError(
        "No fue posible cargar PHONE_NUMBER_ID desde Colab Secrets"
    )

PHONE_NUMBER_ID cargado correctamente


In [ ]:
# ============================================================
# DEFINIR FUNCION DE ENVIO DE RESPUESTA A WHATSAPP
# ============================================================

import requests

META_API_VERSION = "v26.0"

URL_WHATSAPP = (
    f"https://graph.facebook.com/"
    f"{META_API_VERSION}/"
    f"{PHONE_NUMBER_ID}/messages"
)


def enviar_respuesta(numero, texto):
    """
    Envia un mensaje de texto mediante WhatsApp Cloud API.

    Parametros:
        numero: numero del destinatario en formato internacional.
        texto: contenido de la respuesta que se enviara.
    """

    numero = normalizar_numero(numero)
    texto = str(texto).strip()

    if not texto:
        raise ValueError(
            "No se puede enviar una respuesta vacia."
        )

    headers = {
        "Authorization": f"Bearer {WHATSAPP_ACCESS_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "messaging_product": "whatsapp",
        "recipient_type": "individual",
        "to": numero,
        "type": "text",
        "text": {
            "preview_url": False,
            "body": texto
        }
    }

    try:
        respuesta = requests.post(
            URL_WHATSAPP,
            headers=headers,
            json=payload,
            timeout=30
        )

        if respuesta.status_code in (200, 201):
            print("Respuesta enviada correctamente")
            print("Destinatario:", numero)
            return True

        print("No fue posible enviar la respuesta")
        print("Codigo HTTP:", respuesta.status_code)
        print("Detalle:", respuesta.text)

        return False

    except requests.RequestException as error:
        print("Error de comunicacion con WhatsApp Cloud API")
        print("Tipo:", type(error).__name__)
        print("Detalle:", error)

        return False


print("Funcion enviar_respuesta() creada correctamente")
print("WhatsApp Cloud API configurada")


Funcion enviar_respuesta() creada correctamente
WhatsApp Cloud API configurada


In [ ]:
# ============================================================
# PROCESAMIENTO DEL MENSAJE DE WHATSAPP
# ============================================================

def procesar_mensaje_whatsapp(numero, texto):
    """
    Procesa un mensaje recibido desde WhatsApp.

    Flujo:
    1. Normaliza el numero del usuario.
    2. Genera una respuesta utilizando memoria conversacional.
    3. Aplica la regla fija de escalamiento cuando corresponda.
    4. Envia la respuesta mediante WhatsApp Cloud API.
    """

    try:
        numero = normalizar_numero(numero)
        texto = str(texto).strip()

        if not texto:
            print("Mensaje vacio. No se procesa.")
            return False

        print("\nPROCESANDO MENSAJE")
        print("----------------------------------------")
        print("Numero:", numero)
        print("Mensaje:", texto)

        # Generar respuesta con memoria conversacional
        # y regla fija de escalamiento
        respuesta = generar_respuesta_con_memoria(
            numero,
            texto
        )

        print("\nRespuesta generada:")
        print(respuesta)

        # Enviar respuesta a WhatsApp
        resultado_envio = enviar_respuesta(
            numero,
            respuesta
        )

        if resultado_envio:
            print("\nProcesamiento finalizado correctamente")
        else:
            print("\nLa respuesta fue generada, pero no pudo enviarse")

        return resultado_envio

    except Exception as error:
        print("\nError durante el procesamiento del mensaje")
        print("Tipo:", type(error).__name__)
        print("Detalle:", error)

        return False


print("Funcion procesar_mensaje_whatsapp() creada correctamente")
print("Memoria conversacional conectada")
print("Regla fija de escalamiento conectada")
print("Llama Stack conectado")
print("Envio a WhatsApp conectado")


Funcion procesar_mensaje_whatsapp() creada correctamente
Memoria conversacional conectada
Regla fija de escalamiento conectada
Llama Stack conectado
Envio a WhatsApp conectado


### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [ ]:
# ============================================================
# EXPONER EL SERVIDOR FASTAPI CON NGROK
# ============================================================

import subprocess
import threading
import time
import requests

from pyngrok import ngrok


# ------------------------------------------------------------
# 1. CONFIGURAR NGROK
# ------------------------------------------------------------

ngrok.set_auth_token(NGROK_AUTHTOKEN)


# ------------------------------------------------------------
# 2. CERRAR TUNELES NGROK ANTERIORES
# ------------------------------------------------------------

try:
    ngrok.kill()
except Exception:
    pass


# ------------------------------------------------------------
# 3. LEVANTAR FASTAPI CON UVICORN
# ------------------------------------------------------------

def iniciar_servidor():
    import uvicorn

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="warning"
    )


hilo_servidor = threading.Thread(
    target=iniciar_servidor,
    daemon=True
)

hilo_servidor.start()

print("Iniciando servidor FastAPI...")
time.sleep(5)


# ------------------------------------------------------------
# 4. VERIFICAR SERVIDOR LOCAL
# ------------------------------------------------------------

try:
    respuesta_local = requests.get(
        "http://127.0.0.1:8000/docs",
        timeout=10
    )

    print(
        "Servidor local:",
        respuesta_local.status_code
    )

except Exception as error:
    print("No fue posible verificar FastAPI")
    print("Detalle:", error)


# ------------------------------------------------------------
# 5. CREAR TUNEL PUBLICO
# ------------------------------------------------------------

tunel_publico = ngrok.connect(
    8000,
    bind_tls=True
)

URL_PUBLICA = tunel_publico.public_url
CALLBACK_URL = URL_PUBLICA + "/webhook"

print("\nTunel ngrok iniciado correctamente")
print("URL publica:", URL_PUBLICA)
print("Callback URL:", CALLBACK_URL)


Iniciando servidor FastAPI...
Servidor local: 200

Tunel ngrok iniciado correctamente
URL publica: https://width-obstacle-pointing.ngrok-free.dev
Callback URL: https://width-obstacle-pointing.ngrok-free.dev/webhook


### **PROBAR EL FLUJO COMPLETO**

Se manda un primer mensaje mencionando el servicio dental, y un segundo mensaje que depende del primero — para confirmar que el historial sí se está reenviando.

In [ ]:
# ============================================================
# PRUEBA 1 - PRIMER MENSAJE CON CONTEXTO
# ============================================================

import requests
import time

numero_prueba = "529331196350"

payload_1 = {
    "entry": [
        {
            "changes": [
                {
                    "value": {
                        "messages": [
                            {
                                "from": numero_prueba,
                                "type": "text",
                                "text": {
                                    "body": "Quiero una cita dental."
                                }
                            }
                        ]
                    }
                }
            ]
        }
    ]
}

print("PRIMER MENSAJE")
print("----------------------------------------")
print("Numero:", numero_prueba)
print("Mensaje: Quiero una cita dental.")
print("Enviando al webhook local...")

r1 = requests.post(
    "http://127.0.0.1:8000/webhook",
    json=payload_1,
    timeout=30
)

print("Codigo HTTP:", r1.status_code)
print("Respuesta del webhook:", r1.text)

if r1.status_code == 200:
    print("Primer mensaje recibido correctamente por el webhook")
else:
    print("El webhook devolvio un codigo inesperado")


PRIMER MENSAJE
----------------------------------------
Numero: 529331196350
Mensaje: Quiero una cita dental.
Enviando al webhook local...
Mensaje recibido
Numero: 529331196350
Texto: Quiero una cita dental.
Codigo HTTP: 200
Respuesta del webhook: {"status":"ok"}
Primer mensaje recibido correctamente por el webhook


In [ ]:
# ============================================================
# DIAGNOSTICO DEL PROCESAMIENTO
# ============================================================

print("DIAGNOSTICO DE procesar_mensaje_whatsapp")
print("----------------------------------------")

print(
    "Funcion disponible:",
    callable(globals().get("procesar_mensaje_whatsapp"))
)

print(
    "Funcion de memoria disponible:",
    callable(globals().get("generar_respuesta_con_memoria"))
)

print(
    "Funcion de envio disponible:",
    callable(globals().get("enviar_respuesta"))
)

print(
    "Servidor Llama Stack activo:",
    proceso_llama_stack.poll() is None
)

print("\nIniciando prueba directa...")

resultado_diagnostico = procesar_mensaje_whatsapp(
    "529331196350",
    "Quiero una cita dental."
)

print("\nResultado final:", resultado_diagnostico)

DIAGNOSTICO DE procesar_mensaje_whatsapp
----------------------------------------
Funcion disponible: True
Funcion de memoria disponible: True
Funcion de envio disponible: True
Servidor Llama Stack activo: True

Iniciando prueba directa...

PROCESANDO MENSAJE
----------------------------------------
Numero: 529331196350
Mensaje: Quiero una cita dental.

Respuesta generada:
¿Sabes ya qué día te va bien? También me puedes decir el número de teléfono de tu dentista para verificar si hay algún horario disponible. ¿Es el mismo dentista que asististe antes?
No fue posible enviar la respuesta
Codigo HTTP: 401
Detalle: {"error":{"message":"Authentication Error","code":190,"type":"OAuthException","fbtrace_id":"AMwAV3Eh3lsv7pYu0Wsgsol"}}

La respuesta fue generada, pero no pudo enviarse

Resultado final: False


In [ ]:
# ============================================================
# RECARGAR WHATSAPP_ACCESS_TOKEN
# ============================================================

from google.colab import userdata

WHATSAPP_ACCESS_TOKEN = userdata.get("WHATSAPP_ACCESS_TOKEN")

if WHATSAPP_ACCESS_TOKEN:
    print("WHATSAPP_ACCESS_TOKEN recargado correctamente")
else:
    raise RuntimeError(
        "No fue posible cargar WHATSAPP_ACCESS_TOKEN desde Colab Secrets"
    )

WHATSAPP_ACCESS_TOKEN recargado correctamente


In [ ]:
# ============================================================
# PRUEBA CONTROLADA DE ENVIO A WHATSAPP
# ============================================================

numero_destino = "529331196350"

mensaje_prueba = (
    "Prueba de conexion del agente conversacional. "
    "WhatsApp Cloud API configurada correctamente."
)

print("PRUEBA DE ENVIO A WHATSAPP")
print("----------------------------------------")
print("Destinatario:", numero_destino)
print("Enviando mensaje...")

resultado_envio = enviar_respuesta(
    numero_destino,
    mensaje_prueba
)

print("----------------------------------------")
print("Resultado:", resultado_envio)

PRUEBA DE ENVIO A WHATSAPP
----------------------------------------
Destinatario: 529331196350
Enviando mensaje...
Respuesta enviada correctamente
Destinatario: 529331196350
----------------------------------------
Resultado: True


In [ ]:
# ============================================================
# DIAGNOSTICO DE ENVIO - RESPUESTA COMPLETA DE META
# ============================================================

import requests
import json

numero_destino = "529331196350"

headers = {
    "Authorization": f"Bearer {WHATSAPP_ACCESS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "messaging_product": "whatsapp",
    "recipient_type": "individual",
    "to": numero_destino,
    "type": "text",
    "text": {
        "preview_url": False,
        "body": "Mensaje de diagnostico del agente conversacional."
    }
}

respuesta_meta = requests.post(
    URL_WHATSAPP,
    headers=headers,
    json=payload,
    timeout=30
)

print("DIAGNOSTICO DE META")
print("----------------------------------------")
print("Codigo HTTP:", respuesta_meta.status_code)

try:
    datos_meta = respuesta_meta.json()
    print(
        "Respuesta:",
        json.dumps(datos_meta, indent=2, ensure_ascii=False)
    )
except Exception:
    print("Respuesta:", respuesta_meta.text)

DIAGNOSTICO DE META
----------------------------------------
Codigo HTTP: 200
Respuesta: {
  "messaging_product": "whatsapp",
  "contacts": [
    {
      "input": "529331196350",
      "wa_id": "5219331196350"
    }
  ],
  "messages": [
    {
      "id": "wamid.HBgNNTIxOTMzMTE5NjM1MBUCABEYEkZFRUFGMTkwQ0VEMzRERUU2RQA="
    }
  ]
}


In [ ]:
# ============================================================
# PRUEBA DIRECTA CON WA_ID DEVUELTO POR META
# ============================================================

numero_wa_meta = "5219331196350"

payload_wa = {
    "messaging_product": "whatsapp",
    "recipient_type": "individual",
    "to": numero_wa_meta,
    "type": "text",
    "text": {
        "preview_url": False,
        "body": "Prueba directa utilizando el wa_id identificado por Meta."
    }
}

respuesta_wa = requests.post(
    URL_WHATSAPP,
    headers=headers,
    json=payload_wa,
    timeout=30
)

print("PRUEBA CON WA_ID DE META")
print("----------------------------------------")
print("Destinatario solicitado:", numero_wa_meta)
print("Codigo HTTP:", respuesta_wa.status_code)

try:
    datos_wa = respuesta_wa.json()
    print(
        "Respuesta:",
        json.dumps(datos_wa, indent=2, ensure_ascii=False)
    )
except Exception:
    print("Respuesta:", respuesta_wa.text)

PRUEBA CON WA_ID DE META
----------------------------------------
Destinatario solicitado: 5219331196350
Codigo HTTP: 400
Respuesta: {
  "error": {
    "message": "(#131030) Recipient phone number not in allowed list",
    "code": 131030,
    "type": "OAuthException",
    "error_data": {
      "messaging_product": "whatsapp",
      "details": "El número de teléfono del destinatario no está en la lista de autorizados: Agrega el número de teléfono del destinatario a la lista de destinatarios y vuelve a intentarlo."
    },
    "fbtrace_id": "AAoLiJ5dq2OglOxRB8sNhAf"
  }
}


In [ ]:
# ============================================================
# PRUEBA DE ENVIO DESPUES DE VALIDAR DESTINATARIO EN META
# ============================================================

numero_destino = "529331196350"

mensaje_prueba = (
    "Prueba de conexion desde Google Colab. "
    "Destinatario validado previamente en Meta."
)

print("PRUEBA DE ENVIO DESDE COLAB")
print("----------------------------------------")
print("Destinatario:", numero_destino)

resultado_envio = enviar_respuesta(
    numero_destino,
    mensaje_prueba
)

print("----------------------------------------")
print("Resultado:", resultado_envio)

PRUEBA DE ENVIO DESDE COLAB
----------------------------------------
Destinatario: 529331196350
Respuesta enviada correctamente
Destinatario: 529331196350
----------------------------------------
Resultado: True


In [ ]:
# ============================================================
# PRUEBA DE ENVIO DESDE COLAB
# INTERVALO DE CONVERSACION ABIERTO
# ============================================================

numero_destino = "529331196350"

mensaje_prueba = (
    "Prueba de conexion desde Google Colab. "
    "Intervalo de conversacion abierto correctamente."
)

print("PRUEBA DE ENVIO DESDE COLAB")
print("----------------------------------------")
print("Destinatario:", numero_destino)

resultado_envio = enviar_respuesta(
    numero_destino,
    mensaje_prueba
)

print("----------------------------------------")
print("Resultado:", resultado_envio)

PRUEBA DE ENVIO DESDE COLAB
----------------------------------------
Destinatario: 529331196350
Respuesta enviada correctamente
Destinatario: 529331196350
----------------------------------------
Resultado: True


In [ ]:
# ============================================================
# PRUEBA 2 - SEGUNDO MENSAJE DEPENDIENTE DEL PRIMERO
# ============================================================

payload_2 = {
    "entry": [
        {
            "changes": [
                {
                    "value": {
                        "messages": [
                            {
                                "from": "529331196350",
                                "type": "text",
                                "text": {
                                    "body": "¿A que hora abren los sabados?"
                                }
                            }
                        ]
                    }
                }
            ]
        }
    ]
}

print("SEGUNDO MENSAJE")
print("----------------------------------------")
print("Numero: 529331196350")
print("Mensaje: ¿A que hora abren los sabados?")
print("Enviando al webhook local...")

r2 = requests.post(
    "http://127.0.0.1:8000/webhook",
    json=payload_2
)

print("Codigo HTTP:", r2.status_code)
print("Respuesta del webhook:", r2.text)


SEGUNDO MENSAJE
----------------------------------------
Numero: 529331196350
Mensaje: ¿A que hora abren los sabados?
Enviando al webhook local...
Mensaje recibido
Numero: 529331196350
Texto: ¿A que hora abren los sabados?
Codigo HTTP: 200
Respuesta del webhook: {"status":"ok"}


In [ ]:
# ============================================================
# VERIFICAR RESPUESTAS Y CONTINUIDAD DE LA CONVERSACION
# ============================================================

numero_prueba = "529331196350"
historial = obtener_historial(numero_prueba)

print("VERIFICACION DE RESPUESTAS")
print("----------------------------------------")
print("Numero:", numero_prueba)
print("Mensajes almacenados:", len(historial))
print()

for indice, mensaje in enumerate(historial, start=1):
    rol = mensaje.get("role", "")
    contenido = mensaje.get("content", "")

    print(f"Mensaje {indice}")
    print("Rol:", rol)
    print("Contenido:", contenido)
    print("----------------------------------------")

# Comprobacion de continuidad conversacional
texto_historial = " ".join(
    mensaje.get("content", "").lower()
    for mensaje in historial
)

if "dental" in texto_historial or "dentista" in texto_historial or "odontolog" in texto_historial:
    print("La conversacion conserva el contexto del servicio dental")
else:
    print("No se identifico el contexto dental en el historial")


VERIFICACION DE RESPUESTAS
----------------------------------------
Numero: 529331196350
Mensajes almacenados: 10

Mensaje 1
Rol: user
Contenido: Quiero una cita dental.
----------------------------------------
Mensaje 2
Rol: assistant
Contenido: Podría ser 6 a 8 pm, dependiendo de la disponibilidad de tu dentista. ¿Quieres que te ayude a programar una cita en tu agenda o necesitas más información sobre cómo reservar?
----------------------------------------
Mensaje 3
Rol: user
Contenido: ¿A que hora abren los sabados?
----------------------------------------
Mensaje 4
Rol: assistant
Contenido: Por lo general, las consultas de odontología pueden tener un horario específico para los sábados, generalmente al mediodía o por la tarde. Revisaría con tu dentista qué horarios están disponibles para citas de sabado. En la mayoría de las clínicas dentales abren de 9:00 a 13:00 o 14:00 a 18:00 horas en sábados.
----------------------------------------
Mensaje 5
Rol: user
Contenido: Quiero una ci

In [ ]:
# ============================================================
# VER MENSAJES RECIBIDOS POR EL WEBHOOK
# ============================================================

print("MENSAJES RECIBIDOS POR EL WEBHOOK")
print("----------------------------------------")
print("Total de eventos almacenados:", len(mensajes_recibidos))
print()

for indice, evento in enumerate(mensajes_recibidos, start=1):

    try:
        mensaje = (
            evento["entry"][0]
            ["changes"][0]
            ["value"]
            ["messages"][0]
        )

        numero = mensaje.get("from", "")
        tipo = mensaje.get("type", "text")
        texto = mensaje.get("text", {}).get("body", "")

        print(f"Evento {indice}")
        print("Numero:", numero)
        print("Tipo:", tipo)
        print("Mensaje:", texto)
        print("----------------------------------------")

    except (KeyError, IndexError, TypeError):
        print(f"Evento {indice}")
        print("Evento sin mensaje de texto")
        print("----------------------------------------")


MENSAJES RECIBIDOS POR EL WEBHOOK
----------------------------------------
Total de eventos almacenados: 2

Evento 1
Numero: 529331196350
Tipo: text
Mensaje: Quiero una cita dental.
----------------------------------------
Evento 2
Numero: 529331196350
Tipo: text
Mensaje: ¿A que hora abren los sabados?
----------------------------------------


In [ ]:
# ============================================================
# VERIFICAR ESTADO DE LLAMA STACK Y MEMORIA CONVERSACIONAL
# ============================================================

numero_prueba = "529331196350"

print("VERIFICACION DE LLAMA STACK")
print("----------------------------------------")

# Verificar que el cliente de Llama Stack esta disponible
try:
    modelos = cliente_prueba.models.list()

    print("Conexion con Llama Stack: OK")
    print("Modelo utilizado:", MODELO_LLAMA_STACK)

except Exception as error:
    print("Error al consultar Llama Stack:")
    print(error)

print()
print("VERIFICACION DE MEMORIA CONVERSACIONAL")
print("----------------------------------------")

historial = obtener_historial(numero_prueba)

print("Numero:", numero_prueba)
print("Mensajes almacenados:", len(historial))
print("Tipo de almacenamiento: memoria externa en Python")
print()

for indice, mensaje in enumerate(historial, start=1):
    print(
        f"{indice}. "
        f"{mensaje.get('role', '')}: "
        f"{mensaje.get('content', '')}"
    )

print()
print("La memoria conversacional se conserva fuera del modelo")
print("y se envia nuevamente como contexto en cada consulta a Llama 3.1.")


VERIFICACION DE LLAMA STACK
----------------------------------------
Conexion con Llama Stack: OK
Modelo utilizado: ollama/llama3.1:8b

VERIFICACION DE MEMORIA CONVERSACIONAL
----------------------------------------
Numero: 529331196350
Mensajes almacenados: 10
Tipo de almacenamiento: memoria externa en Python

1. user: Quiero una cita dental.
2. assistant: Podría ser 6 a 8 pm, dependiendo de la disponibilidad de tu dentista. ¿Quieres que te ayude a programar una cita en tu agenda o necesitas más información sobre cómo reservar?
3. user: ¿A que hora abren los sabados?
4. assistant: Por lo general, las consultas de odontología pueden tener un horario específico para los sábados, generalmente al mediodía o por la tarde. Revisaría con tu dentista qué horarios están disponibles para citas de sabado. En la mayoría de las clínicas dentales abren de 9:00 a 13:00 o 14:00 a 18:00 horas en sábados.
5. user: Quiero una cita dental.
6. assistant: ¡Genial! ¿Qué día y horario te gustaría? Eso te ayu

In [ ]:
# ============================================================
# FINALIZAR PROCESO DE TUNELIZACION
# ============================================================

from pyngrok import ngrok

print("FINALIZACION DEL TUNEL NGROK")
print("----------------------------------------")

try:
    tuneles_activos = ngrok.get_tunnels()

    if tuneles_activos:
        print("Tuneles activos encontrados:", len(tuneles_activos))

        for tunel in tuneles_activos:
            print("Cerrando:", tunel.public_url)
            ngrok.disconnect(tunel.public_url)

        ngrok.kill()

        print()
        print("Tunel ngrok finalizado correctamente.")
        print("El webhook publico ya no se encuentra disponible.")

    else:
        print("No se encontraron tuneles ngrok activos.")

except Exception as error:
    print("Se produjo un error al finalizar ngrok:")
    print(error)


FINALIZACION DEL TUNEL NGROK
----------------------------------------
Tuneles activos encontrados: 1
Cerrando: https://width-obstacle-pointing.ngrok-free.dev

Tunel ngrok finalizado correctamente.
El webhook publico ya no se encuentra disponible.


## **CHALLENGE: AGENTE DE UNA VETERINARIA**

Una vez visto el ***Hands-On: Diseño de Agentes Conversacionales***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un agente similar, pero para una veterinaria: además de la regla de escalamiento a una persona, se agrega una regla de urgencia médica, y un límite al historial que se conserva en la conversación de Llama Stack.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab, descarga el modelo Llama 3.1 8B, instala y levanta **Llama Stack** con Ollama registrado, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [1]:
# ============================================================
# CHALLENGE VETERINARIA - INSTALAR OLLAMA
# ============================================================

import subprocess

print("Verificando dependencia zstd...")

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "zstd"],
    check=True
)

print("zstd instalado correctamente.")
print("Instalando Ollama...")

resultado = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    check=False
)

if resultado.returncode != 0:
    raise RuntimeError("No fue posible instalar Ollama.")

print("Ollama instalado correctamente.")


Verificando dependencia zstd...
zstd instalado correctamente.
Instalando Ollama...
Ollama instalado correctamente.


In [2]:
# ============================================================
# CHALLENGE VETERINARIA - INICIAR SERVIDOR OLLAMA
# ============================================================

import subprocess
import time
import requests

print("Iniciando servidor de Ollama...")

proceso_ollama = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

try:
    respuesta = requests.get(
        "http://127.0.0.1:11434/api/tags",
        timeout=10
    )

    if respuesta.status_code == 200:
        print("Servidor Ollama activo.")
        print("Direccion: http://127.0.0.1:11434")
    else:
        print("Ollama respondio con codigo:", respuesta.status_code)

except Exception as error:
    print("No fue posible conectar con Ollama.")
    print("Error:", error)


Iniciando servidor de Ollama...
Servidor Ollama activo.
Direccion: http://127.0.0.1:11434


In [3]:
# ============================================================
# CHALLENGE VETERINARIA - INSTALAR DEPENDENCIAS
# ============================================================

!pip install -q fastapi uvicorn requests pyngrok ollama pyyaml

print("Dependencias instaladas correctamente.")


Dependencias instaladas correctamente.


In [4]:
# ============================================================
# CHALLENGE VETERINARIA - LLAMA STACK
# ============================================================

import subprocess
import time
import requests

print("Instalando Llama Stack...")

subprocess.run(
    [
        "pip", "install", "-q",
        "llama-stack==0.7.3",
        "llama-stack-client==0.7.5",
        "mcp==1.30.0"
    ],
    check=True
)

print("Llama Stack instalado correctamente.")

print("Descargando Llama 3.1 8B si es necesario...")

resultado = subprocess.run(
    ["ollama", "pull", "llama3.1:8b"],
    check=False
)

if resultado.returncode != 0:
    raise RuntimeError("No fue posible descargar llama3.1:8b.")

print("Modelo llama3.1:8b disponible.")


Instalando Llama Stack...
Llama Stack instalado correctamente.
Descargando Llama 3.1 8B si es necesario...
Modelo llama3.1:8b disponible.


In [5]:
# ============================================================
# LEVANTAR LLAMA STACK CON OLLAMA REGISTRADO
# ============================================================

import os
import subprocess
import time
import requests

config_llama_stack = """
version: '2'

image_name: starter
apis:
  - models
  - inference

providers:
  inference:
    - provider_id: ollama
      provider_type: remote::ollama
      config:
        url: http://127.0.0.1:11434

models:
  - model_id: ollama/llama3.1:8b
    provider_id: ollama
    provider_model_id: llama3.1:8b
"""

ruta_config = "/content/mi_config_llama_stack.yaml"

with open(ruta_config, "w", encoding="utf-8") as archivo:
    archivo.write(config_llama_stack)

print("Configuracion creada:", ruta_config)

proceso_llama_stack = subprocess.Popen(
    [
        "llama", "stack", "run",
        ruta_config,
        "--port", "8321"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Iniciando servidor Llama Stack...")
time.sleep(10)

try:
    respuesta = requests.get(
        "http://127.0.0.1:8321/v1/models",
        timeout=15
    )

    if respuesta.status_code == 200:
        print("Servidor Llama Stack activo.")
        print("Direccion: http://127.0.0.1:8321")
        print("Modelo registrado: ollama/llama3.1:8b")
    else:
        print("Llama Stack respondio con codigo:", respuesta.status_code)
        print(respuesta.text)

except Exception as error:
    print("No fue posible conectar con Llama Stack.")
    print("Error:", error)

Configuracion creada: /content/mi_config_llama_stack.yaml
Iniciando servidor Llama Stack...
No fue posible conectar con Llama Stack.
Error: HTTPConnectionPool(host='127.0.0.1', port=8321): Max retries exceeded with url: /v1/models (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b44752c9e50>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [6]:
# ============================================================
# DIAGNOSTICO DE LLAMA STACK
# ============================================================

import subprocess

print("DIAGNOSTICO DE LLAMA STACK")
print("----------------------------------------")

resultado = subprocess.run(
    [
        "llama", "stack", "run",
        "/content/mi_config_llama_stack.yaml",
        "--port", "8321"
    ],
    capture_output=True,
    text=True,
    timeout=30
)

print("Codigo de salida:", resultado.returncode)

print("\nSALIDA:")
print(resultado.stdout)

print("\nERRORES:")
print(resultado.stderr)

DIAGNOSTICO DE LLAMA STACK
----------------------------------------
Codigo de salida: 0

SALIDA:


ERRORES:
INFO     2026-09-28T18:57:00.211273Z  llama_stack.cli.stack.run:108 Using stack configuration   category=cli            
         config_file=/content/mi_config_llama_stack.yaml                                                                
INFO     2026-09-28T18:57:00.385735Z  llama_stack.cli.stack.run:172 HTTPS enabled with certificates   category=cli      
         certfile=None keyfile=None                                                                                     
INFO     2026-09-28T18:57:00.387706Z  llama_stack.cli.stack.run:174 Listening on   category=cli host= port=8321         
INFO     2026-09-28T18:57:00.654887Z  llama_stack.core.server.server:374 Stack Configuration:   category=core::server   
INFO     2026-09-28T18:57:00.659483Z  llama_stack.core.server.server:377 apis:                                          
         - models                            

In [7]:
# ============================================================
# COMPROBAR LLAMA STACK ACTIVO EN EL PUERTO 8321
# ============================================================

import requests

print("COMPROBACION DE LLAMA STACK")
print("----------------------------------------")

try:
    respuesta = requests.get(
        "http://127.0.0.1:8321/v1/models",
        timeout=15
    )

    print("Codigo HTTP:", respuesta.status_code)

    if respuesta.status_code == 200:
        datos = respuesta.json()

        print("Servidor Llama Stack: ACTIVO")
        print("Direccion: http://127.0.0.1:8321")
        print()
        print("Modelos registrados:")

        for modelo in datos.get("data", []):
            print("-", modelo.get("id"))

    else:
        print("El puerto responde, pero no correctamente.")
        print(respuesta.text)

except Exception as error:
    print("No fue posible consultar Llama Stack.")
    print("Error:", error)

COMPROBACION DE LLAMA STACK
----------------------------------------
Codigo HTTP: 200
Servidor Llama Stack: ACTIVO
Direccion: http://127.0.0.1:8321

Modelos registrados:
- ollama/llama3.1:8b


In [8]:
# ============================================================
# CHALLENGE VETERINARIA - LEER CREDENCIALES DE COLAB SECRETS
# ============================================================

from google.colab import userdata

print("Cargando credenciales desde Colab Secrets...")

WHATSAPP_ACCESS_TOKEN = userdata.get("WHATSAPP_ACCESS_TOKEN")
VERIFY_TOKEN = userdata.get("VERIFY_TOKEN")
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
PHONE_NUMBER_ID = userdata.get("PHONE_NUMBER_ID")

credenciales = {
    "WHATSAPP_ACCESS_TOKEN": WHATSAPP_ACCESS_TOKEN,
    "VERIFY_TOKEN": VERIFY_TOKEN,
    "NGROK_AUTHTOKEN": NGROK_AUTHTOKEN,
    "PHONE_NUMBER_ID": PHONE_NUMBER_ID
}

for nombre, valor in credenciales.items():
    if valor:
        print(nombre, ": cargado correctamente")
    else:
        print(nombre, ": NO DISPONIBLE")

if not all(credenciales.values()):
    raise RuntimeError(
        "Falta una o mas credenciales en Colab Secrets."
    )

print()
print("Todas las credenciales fueron cargadas correctamente.")


Cargando credenciales desde Colab Secrets...
WHATSAPP_ACCESS_TOKEN : cargado correctamente
VERIFY_TOKEN : cargado correctamente
NGROK_AUTHTOKEN : cargado correctamente
PHONE_NUMBER_ID : cargado correctamente

Todas las credenciales fueron cargadas correctamente.


**2. Construye la memoria de conversación:** Crea una conversación de **Llama Stack** por número de teléfono (`client.conversations.create()`) con una memoria de respuesta.

In [9]:
# ============================================================
# CONVERSACION DE LLAMA STACK POR NUMERO DE TELEFONO
# ============================================================

from llama_stack_client import LlamaStackClient

client = LlamaStackClient(
    base_url="http://127.0.0.1:8321",
    timeout=300.0
)

# Diccionario para asociar cada numero de telefono
# con su conversacion de Llama Stack
conversaciones_por_numero = {}

def obtener_conversacion(numero):
    numero = str(numero).strip()

    if numero not in conversaciones_por_numero:

        conversacion = client.conversations.create()

        conversaciones_por_numero[numero] = {
            "conversation_id": conversacion.id,
            "historial": []
        }

        print("Nueva conversacion creada")
        print("Numero:", numero)
        print("Conversation ID:", conversacion.id)

    return conversaciones_por_numero[numero]


print("Sistema de conversaciones por numero creado correctamente.")


Sistema de conversaciones por numero creado correctamente.


In [10]:
# ============================================================
# GENERACION DE RESPUESTA CON MEMORIA
# ============================================================

MODELO_LLAMA_STACK = "ollama/llama3.1:8b"
MAX_MENSAJES_HISTORIAL = 10

def generar_respuesta_con_memoria(numero, mensaje_usuario):

    numero = str(numero).strip()
    mensaje_usuario = str(mensaje_usuario).strip()

    if not mensaje_usuario:
        raise ValueError("El mensaje del usuario no puede estar vacio.")

    # Obtener o crear la conversacion asociada al numero
    memoria = obtener_conversacion(numero)
    historial = memoria["historial"]

    # Construir contexto para Llama
    mensajes = [
        {
            "role": "system",
            "content": (
                "Eres el asistente virtual de una veterinaria. "
                "Responde de forma clara, breve y profesional. "
                "Utiliza el historial de la conversacion para "
                "mantener el contexto entre mensajes."
            )
        }
    ]

    mensajes.extend(historial)

    mensajes.append(
        {
            "role": "user",
            "content": mensaje_usuario
        }
    )

    # Consultar Llama 3.1 mediante Llama Stack
    respuesta = client.chat.completions.create(
        model=MODELO_LLAMA_STACK,
        messages=mensajes,
        max_tokens=150
    )

    texto_respuesta = respuesta.choices[0].message.content.strip()

    # Guardar nuevo turno en el historial
    historial.append(
        {
            "role": "user",
            "content": mensaje_usuario
        }
    )

    historial.append(
        {
            "role": "assistant",
            "content": texto_respuesta
        }
    )

    # Conservar solamente los ultimos mensajes
    if len(historial) > MAX_MENSAJES_HISTORIAL:
        memoria["historial"] = historial[-MAX_MENSAJES_HISTORIAL:]

    return texto_respuesta


print("Funcion generar_respuesta_con_memoria() creada correctamente.")
print("Limite del historial:", MAX_MENSAJES_HISTORIAL, "mensajes")


Funcion generar_respuesta_con_memoria() creada correctamente.
Limite del historial: 10 mensajes


**3. Agrega la regla fija y protege el procesamiento por número:**

   * Cuando el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento.
   * Igual que en el Hands-On, usa un lock por número de teléfono para que dos mensajes seguidos del mismo usuario no disparen dos llamadas en paralelo sobre la misma conversación.

In [11]:
# ============================================================
# LOCK POR NUMERO DE TELEFONO
# ============================================================

import threading

# Diccionario de locks asociados a cada numero de telefono
locks_por_numero = {}

# Lock general para proteger la creacion de nuevos locks
lock_general = threading.Lock()


def obtener_lock(numero):
    numero = str(numero).strip()

    with lock_general:
        if numero not in locks_por_numero:
            locks_por_numero[numero] = threading.Lock()

    return locks_por_numero[numero]


print("Sistema de locks por numero creado correctamente.")


Sistema de locks por numero creado correctamente.


In [18]:
# ============================================================
# PROCESAMIENTO DEL AGENTE DE LA VETERINARIA
# ============================================================

def solicita_atencion_humana(texto):
    texto = str(texto).lower().strip()

    expresiones_escalamiento = [
        "hablar con una persona",
        "hablar con alguien",
        "hablar con un humano",
        "hablar con un asesor",
        "hablar con un veterinario",
        "quiero una persona",
        "quiero un asesor",
        "quiero un veterinario"
    ]

    return any(
        expresion in texto
        for expresion in expresiones_escalamiento
    )


def es_urgencia_veterinaria(texto):
    texto = str(texto).lower().strip()

    palabras_urgencia = [
        "urgencia",
        "emergencia",
        "sangrado",
        "convulsion",
        "convulsiones",
        "no respira",
        "atropellado",
        "envenenamiento",
        "veneno"
    ]

    return any(
        palabra in texto
        for palabra in palabras_urgencia
    )


def procesar_mensaje(numero, texto):
    numero = normalizar_numero(numero)
    texto = str(texto).strip()

    if not texto:
        raise ValueError("El mensaje no puede estar vacio.")

    lock_usuario = obtener_lock(numero)

    with lock_usuario:

        # Regla fija de urgencia veterinaria
        if es_urgencia_veterinaria(texto):
            respuesta = (
                "Tu mensaje indica una posible urgencia veterinaria. "
                "Busca atencion veterinaria inmediata y comunicate "
                "directamente con la clinica o servicio de emergencias."
            )

        # Regla fija de escalamiento a una persona
        elif solicita_atencion_humana(texto):
            respuesta = (
                "Claro. Voy a escalar tu solicitud para que "
                "una persona del equipo de la veterinaria "
                "pueda atenderte."
            )

        # Respuesta normal mediante Llama + memoria
        else:
            respuesta = generar_respuesta_con_memoria(
                numero,
                texto
            )

        print("Respuesta generada:")
        print(respuesta)

        # Enviar la respuesta a WhatsApp
        resultado_envio = enviar_respuesta(
            numero,
            respuesta
        )

        return respuesta


print("Funcion procesar_mensaje() actualizada correctamente.")
print("Regla de urgencia veterinaria: configurada.")
print("Regla de escalamiento humano: configurada.")
print("Respuesta por WhatsApp: integrada.")
print("Procesamiento protegido por numero: activo.")

Funcion procesar_mensaje() actualizada correctamente.
Regla de urgencia veterinaria: configurada.
Regla de escalamiento humano: configurada.
Respuesta por WhatsApp: integrada.
Procesamiento protegido por numero: activo.


**4. Define los endpoints del webhook:** usa `BackgroundTasks` para responder de inmediato a Meta, delegando el procesamiento real a la función anterior.

In [13]:
# ============================================================
# CONFIGURAR EL SERVIDOR DEL WEBHOOK
# ============================================================

from fastapi import FastAPI, Request, BackgroundTasks
from fastapi.responses import PlainTextResponse, JSONResponse

app = FastAPI()

# Registro de eventos recibidos para las pruebas del Challenge
mensajes_recibidos = []

print("Servidor FastAPI configurado correctamente.")
print("Aplicacion webhook creada.")


Servidor FastAPI configurado correctamente.
Aplicacion webhook creada.


In [14]:
# ============================================================
# DEFINIR FUNCION DE NORMALIZACION DE NUMERO
# ============================================================

import re

def normalizar_numero(numero):
    """
    Normaliza numeros telefonicos para WhatsApp.

    - Elimina espacios, guiones, parentesis y el simbolo +.
    - Convierte numeros mexicanos de 10 digitos a 52XXXXXXXXXX.
    - Corrige el formato antiguo 521XXXXXXXXXX a 52XXXXXXXXXX.
    """

    # Conservar unicamente digitos
    numero = re.sub(r"\D", "", str(numero))

    # Formato antiguo de Mexico: 521 + 10 digitos
    if numero.startswith("521") and len(numero) == 13:
        numero = "52" + numero[3:]

    # Numero mexicano nacional de 10 digitos
    elif len(numero) == 10:
        numero = "52" + numero

    # Validacion basica
    if not numero.isdigit() or len(numero) < 10 or len(numero) > 15:
        raise ValueError(
            f"Numero telefonico no valido para WhatsApp: {numero}"
        )

    return numero


# Prueba controlada
numero_prueba = "933 119 6350"
numero_normalizado = normalizar_numero(numero_prueba)

print("Funcion normalizar_numero() creada correctamente.")
print("Numero de prueba:", numero_prueba)
print("Numero normalizado:", numero_normalizado)

assert numero_normalizado == "529331196350"

print("Prueba de normalizacion superada.")


Funcion normalizar_numero() creada correctamente.
Numero de prueba: 933 119 6350
Numero normalizado: 529331196350
Prueba de normalizacion superada.


In [15]:
# ============================================================
# ENDPOINT DE VERIFICACION DE CREDENCIALES
# ============================================================

@app.get("/webhook")
async def verificar_webhook(request: Request):

    mode = request.query_params.get("hub.mode")
    token = request.query_params.get("hub.verify_token")
    challenge = request.query_params.get("hub.challenge")

    print("Solicitud de verificacion recibida")
    print("Modo:", mode)

    if mode == "subscribe" and token == VERIFY_TOKEN:
        print("Verificacion correcta")
        return PlainTextResponse(
            content=challenge,
            status_code=200
        )

    print("Verificacion rechazada")

    return PlainTextResponse(
        content="Forbidden",
        status_code=403
    )


print("Endpoint GET /webhook creado correctamente.")


Endpoint GET /webhook creado correctamente.


In [16]:
# ============================================================
# ENDPOINT DE RECEPCION DE MENSAJES
# ============================================================

@app.post("/webhook")
async def recibir_mensaje(
    request: Request,
    background_tasks: BackgroundTasks
):

    try:
        datos = await request.json()

        # Guardar evento para las pruebas posteriores
        mensajes_recibidos.append(datos)

        entry = datos.get("entry", [])

        if not entry:
            return JSONResponse(
                content={"status": "ok"},
                status_code=200
            )

        changes = entry[0].get("changes", [])

        if not changes:
            return JSONResponse(
                content={"status": "ok"},
                status_code=200
            )

        value = changes[0].get("value", {})
        messages = value.get("messages", [])

        if not messages:
            return JSONResponse(
                content={"status": "ok"},
                status_code=200
            )

        mensaje = messages[0]

        numero = mensaje.get("from", "")
        tipo = mensaje.get("type", "")

        if tipo == "text":

            texto = (
                mensaje.get("text", {})
                .get("body", "")
                .strip()
            )

            numero = normalizar_numero(numero)

            print("Mensaje recibido")
            print("Numero:", numero)
            print("Texto:", texto)

            if texto:
                background_tasks.add_task(
                    procesar_mensaje,
                    numero,
                    texto
                )

        # Meta recibe inmediatamente HTTP 200
        return JSONResponse(
            content={"status": "ok"},
            status_code=200
        )

    except Exception as error:

        print("Error al recibir mensaje:")
        print(error)

        # Se responde 200 para evitar reintentos innecesarios de Meta
        return JSONResponse(
            content={"status": "error"},
            status_code=200
        )


print("Endpoint POST /webhook creado correctamente.")
print("Procesamiento delegado mediante BackgroundTasks.")


Endpoint POST /webhook creado correctamente.
Procesamiento delegado mediante BackgroundTasks.


In [17]:
# ============================================================
# FUNCION DE ENVIO DE RESPUESTA A WHATSAPP
# ============================================================

import requests

META_API_VERSION = "v26.0"

URL_WHATSAPP = (
    f"https://graph.facebook.com/"
    f"{META_API_VERSION}/"
    f"{PHONE_NUMBER_ID}/messages"
)


def enviar_respuesta(numero, texto):

    numero = normalizar_numero(numero)
    texto = str(texto).strip()

    if not texto:
        raise ValueError(
            "La respuesta que se enviara a WhatsApp no puede estar vacia."
        )

    headers = {
        "Authorization": f"Bearer {WHATSAPP_ACCESS_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {
            "body": texto
        }
    }

    try:
        respuesta = requests.post(
            URL_WHATSAPP,
            headers=headers,
            json=payload,
            timeout=30
        )

        if respuesta.status_code in (200, 201):

            print("Respuesta enviada correctamente.")
            print("Destinatario:", numero)

            return True

        print("WhatsApp rechazo el envio.")
        print("Codigo HTTP:", respuesta.status_code)
        print("Respuesta:", respuesta.text)

        return False

    except requests.RequestException as error:

        print("Error de comunicacion con WhatsApp:")
        print(error)

        return False


print("Funcion enviar_respuesta() creada correctamente.")
print("WhatsApp Cloud API configurada.")


Funcion enviar_respuesta() creada correctamente.
WhatsApp Cloud API configurada.


**5. Expón el servidor con ngrok y prueba el flujo completo:**

   * Manda una conversación de al menos 4 mensajes y confirma que el historial se recorta correctamente.
   * Manda un mensaje con una palabra de urgencia y confirma que dispara la regla fija en vez de una respuesta de Llama.

In [19]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - EXPONER EL SERVIDOR Y PROBAR EL FLUJO COMPLETO
# CELDA 1 - INSTALAR E INICIAR TUNEL PUBLICO CON NGROK
# ============================================================

import threading
import time
import uvicorn

from pyngrok import ngrok

print("Configurando ngrok...")

# ------------------------------------------------------------
# 1. CONFIGURAR CREDENCIALES DE NGROK
# ------------------------------------------------------------

ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Cerrar tuneles anteriores si existieran
ngrok.kill()


# ------------------------------------------------------------
# 2. INICIAR SERVIDOR FASTAPI EN SEGUNDO PLANO
# ------------------------------------------------------------

def iniciar_servidor():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="warning"
    )


hilo_servidor = threading.Thread(
    target=iniciar_servidor,
    daemon=True
)

hilo_servidor.start()

time.sleep(3)


# ------------------------------------------------------------
# 3. CREAR TUNEL PUBLICO CON NGROK
# ------------------------------------------------------------

tunel_publico = ngrok.connect(
    8000,
    bind_tls=True
)

URL_PUBLICA = tunel_publico.public_url
CALLBACK_URL = URL_PUBLICA + "/webhook"


# ------------------------------------------------------------
# 4. MOSTRAR DIRECCIONES DEL SERVIDOR Y WEBHOOK
# ------------------------------------------------------------

print()
print("Servidor FastAPI activo en:")
print("http://127.0.0.1:8000")

print()
print("Tunel ngrok activo:")
print(URL_PUBLICA)

print()
print("Callback URL:")
print(CALLBACK_URL)


Configurando ngrok...

Servidor FastAPI activo en:
http://127.0.0.1:8000

Tunel ngrok activo:
https://width-obstacle-pointing.ngrok-free.dev

Callback URL:
https://width-obstacle-pointing.ngrok-free.dev/webhook


In [20]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - EXPONER EL SERVIDOR Y PROBAR EL FLUJO COMPLETO
# CELDA 2 - MANDAR EL PRIMER MENSAJE MENCIONANDO EL SERVICIO
# ============================================================

import requests

# ------------------------------------------------------------
# 1. DEFINIR MENSAJE DE PRUEBA
# ------------------------------------------------------------

numero_prueba = "529331196350"
mensaje_prueba_1 = "Quiero una cita para vacunar a mi perro."


# ------------------------------------------------------------
# 2. CONSTRUIR PAYLOAD SIMULANDO WHATSAPP
# ------------------------------------------------------------

payload_1 = {
    "entry": [
        {
            "changes": [
                {
                    "value": {
                        "messages": [
                            {
                                "from": numero_prueba,
                                "type": "text",
                                "text": {
                                    "body": mensaje_prueba_1
                                }
                            }
                        ]
                    }
                }
            ]
        }
    ]
}


# ------------------------------------------------------------
# 3. ENVIAR MENSAJE AL WEBHOOK LOCAL
# ------------------------------------------------------------

print("PRIMER MENSAJE")
print("----------------------------------------")
print("Numero:", numero_prueba)
print("Mensaje:", mensaje_prueba_1)
print("Enviando al webhook local...")

respuesta_prueba_1 = requests.post(
    "http://127.0.0.1:8000/webhook",
    json=payload_1,
    timeout=30
)


# ------------------------------------------------------------
# 4. MOSTRAR RESULTADO DEL WEBHOOK
# ------------------------------------------------------------

print()
print("Codigo HTTP:", respuesta_prueba_1.status_code)
print("Respuesta del webhook:", respuesta_prueba_1.text)

if respuesta_prueba_1.status_code == 200:
    print("Primer mensaje recibido correctamente por el webhook.")
else:
    print("El webhook no proceso correctamente el primer mensaje.")


PRIMER MENSAJE
----------------------------------------
Numero: 529331196350
Mensaje: Quiero una cita para vacunar a mi perro.
Enviando al webhook local...
Mensaje recibido
Numero: 529331196350
Texto: Quiero una cita para vacunar a mi perro.

Codigo HTTP: 200
Respuesta del webhook: {"status":"ok"}
Primer mensaje recibido correctamente por el webhook.


In [21]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# DIAGNOSTICO - PROCESAMIENTO DIRECTO DEL PRIMER MENSAJE
# ============================================================

print("DIAGNOSTICO DE PROCESAMIENTO")
print("----------------------------------------")

resultado = procesar_mensaje(
    "529331196350",
    "Quiero una cita para vacunar a mi perro."
)

print()
print("Resultado devuelto:")
print(resultado)

DIAGNOSTICO DE PROCESAMIENTO
----------------------------------------
Respuesta generada:
Me alegra que hayas pensado en la salud de tu perro. ¿Cuál es la edad y raza de tu perro? Eso me ayudará a asesorarte sobre las vacunas necesarias. También necesitaré el nombre y la descripción de todas las vacunas que ya tiene tu perro para asegurarme de que no se repitan.

En cuanto a la disponibilidad de la veterinaria, tenemos citas disponibles el lunes a las 10:00, martes a las 14:30 y jueves a las 18:00. ¿Cuál de estas opciones te parece mejor?
WhatsApp rechazo el envio.
Codigo HTTP: 401
Respuesta: {"error":{"message":"Authentication Error","code":190,"type":"OAuthException","fbtrace_id":"ATYVOZe5Xq7BhvTHhH-txrA"}}

Resultado devuelto:
Me alegra que hayas pensado en la salud de tu perro. ¿Cuál es la edad y raza de tu perro? Eso me ayudará a asesorarte sobre las vacunas necesarias. También necesitaré el nombre y la descripción de todas las vacunas que ya tiene tu perro para asegurarme de que 

In [22]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# DIAGNOSTICO - RECARGAR TOKEN DE WHATSAPP CLOUD API
# ============================================================

from google.colab import userdata

# ------------------------------------------------------------
# 1. RECARGAR TOKEN DESDE COLAB SECRETS
# ------------------------------------------------------------

WHATSAPP_ACCESS_TOKEN = userdata.get("WHATSAPP_ACCESS_TOKEN")


# ------------------------------------------------------------
# 2. VALIDAR QUE EL TOKEN FUE CARGADO
# ------------------------------------------------------------

if WHATSAPP_ACCESS_TOKEN:
    print("WHATSAPP_ACCESS_TOKEN recargado correctamente.")
    print("Token disponible en la sesion actual.")
else:
    print("ERROR: No fue posible cargar WHATSAPP_ACCESS_TOKEN.")

WHATSAPP_ACCESS_TOKEN recargado correctamente.
Token disponible en la sesion actual.


In [23]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# DIAGNOSTICO - PRUEBA DIRECTA DE ENVIO A WHATSAPP
# ============================================================

# ------------------------------------------------------------
# 1. DEFINIR DESTINATARIO Y MENSAJE DE CONTROL
# ------------------------------------------------------------

numero_prueba = "529331196350"

mensaje_control = (
    "Prueba de conexion del agente de la veterinaria. "
    "WhatsApp Cloud API funcionando correctamente."
)


# ------------------------------------------------------------
# 2. ENVIAR MENSAJE DIRECTAMENTE A WHATSAPP
# ------------------------------------------------------------

print("PRUEBA DIRECTA DE ENVIO")
print("----------------------------------------")
print("Destinatario:", numero_prueba)

resultado_envio = enviar_respuesta(
    numero_prueba,
    mensaje_control
)


# ------------------------------------------------------------
# 3. MOSTRAR RESULTADO
# ------------------------------------------------------------

print()
print("Resultado del envio:", resultado_envio)

if resultado_envio:
    print("Prueba de WhatsApp completada correctamente.")
else:
    print("La prueba de WhatsApp no fue completada.")

PRUEBA DIRECTA DE ENVIO
----------------------------------------
Destinatario: 529331196350
Respuesta enviada correctamente.
Destinatario: 529331196350

Resultado del envio: True
Prueba de WhatsApp completada correctamente.


In [24]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - EXPONER EL SERVIDOR Y PROBAR EL FLUJO COMPLETO
# CELDA 3 - SEGUNDO MENSAJE DEPENDIENTE DEL PRIMERO
# ============================================================

import requests

# ------------------------------------------------------------
# 1. DEFINIR SEGUNDO MENSAJE
# ------------------------------------------------------------

numero_prueba = "529331196350"
mensaje_prueba_2 = "Tiene 3 años y es un labrador. ¿Que necesito llevar?"


# ------------------------------------------------------------
# 2. CONSTRUIR PAYLOAD SIMULANDO WHATSAPP
# ------------------------------------------------------------

payload_2 = {
    "entry": [
        {
            "changes": [
                {
                    "value": {
                        "messages": [
                            {
                                "from": numero_prueba,
                                "type": "text",
                                "text": {
                                    "body": mensaje_prueba_2
                                }
                            }
                        ]
                    }
                }
            ]
        }
    ]
}


# ------------------------------------------------------------
# 3. ENVIAR SEGUNDO MENSAJE AL WEBHOOK
# ------------------------------------------------------------

print("SEGUNDO MENSAJE")
print("----------------------------------------")
print("Numero:", numero_prueba)
print("Mensaje:", mensaje_prueba_2)
print("Enviando al webhook...")

respuesta_prueba_2 = requests.post(
    "http://127.0.0.1:8000/webhook",
    json=payload_2,
    timeout=30
)


# ------------------------------------------------------------
# 4. MOSTRAR RESULTADO DEL WEBHOOK
# ------------------------------------------------------------

print()
print("Codigo HTTP:", respuesta_prueba_2.status_code)
print("Respuesta del webhook:", respuesta_prueba_2.text)

if respuesta_prueba_2.status_code == 200:
    print("Segundo mensaje recibido correctamente por el webhook.")
else:
    print("El webhook no proceso correctamente el segundo mensaje.")

SEGUNDO MENSAJE
----------------------------------------
Numero: 529331196350
Mensaje: Tiene 3 años y es un labrador. ¿Que necesito llevar?
Enviando al webhook...
Mensaje recibido
Numero: 529331196350
Texto: Tiene 3 años y es un labrador. ¿Que necesito llevar?

Codigo HTTP: 200
Respuesta del webhook: {"status":"ok"}
Segundo mensaje recibido correctamente por el webhook.


In [25]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - DIAGNOSTICO DEL SEGUNDO MENSAJE
# CELDA DE INSPECCION - ESTADO DE MEMORIA
# ============================================================

numero_prueba = "529331196350"

print("ESTADO ACTUAL DE LA CONVERSACION")
print("----------------------------------------")

if numero_prueba in conversaciones_por_numero:

    memoria = conversaciones_por_numero[numero_prueba]

    print("Conversation ID:")
    print(memoria["conversation_id"])

    print()
    print("Cantidad de mensajes almacenados:")
    print(len(memoria["historial"]))

    print()
    print("Historial actual:")
    print("----------------------------------------")

    for indice, mensaje in enumerate(memoria["historial"], start=1):
        print(
            indice,
            "|",
            mensaje["role"],
            "|",
            mensaje["content"]
        )

else:
    print("No existe una conversacion almacenada para este numero.")

ESTADO ACTUAL DE LA CONVERSACION
----------------------------------------
Conversation ID:
conv_2640f0752be6f65464ed837be7927b6dd473cfe51472b30c

Cantidad de mensajes almacenados:
4

Historial actual:
----------------------------------------
1 | user | Quiero una cita para vacunar a mi perro.
2 | assistant | Me alegra que hayas pensado en la salud de tu perro. ¿Tienes información sobre la raza y edad de tu perro? Esto me ayudará a asesorarte sobre las vacunas nécessarias. ¿Tienes preferencia para la fecha y hora de la cita o prefieres que te lo asigne yo?
3 | user | Quiero una cita para vacunar a mi perro.
4 | assistant | Me alegra que hayas pensado en la salud de tu perro. ¿Cuál es la edad y raza de tu perro? Eso me ayudará a asesorarte sobre las vacunas necesarias. También necesitaré el nombre y la descripción de todas las vacunas que ya tiene tu perro para asegurarme de que no se repitan.

En cuanto a la disponibilidad de la veterinaria, tenemos citas disponibles el lunes a las 10:0

In [26]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - PRUEBA CONTROLADA DEL SEGUNDO MENSAJE
# CELDA DE DIAGNOSTICO - MEMORIA + LLAMA + WHATSAPP
# ============================================================

numero_prueba = "529331196350"

mensaje_prueba_2 = (
    "Tiene 3 años y es un labrador. "
    "¿Que necesito llevar?"
)

# ------------------------------------------------------------
# 1. PROCESAR DIRECTAMENTE EL SEGUNDO MENSAJE
# ------------------------------------------------------------

print("PROCESANDO SEGUNDO MENSAJE")
print("----------------------------------------")
print("Numero:", numero_prueba)
print("Mensaje:", mensaje_prueba_2)
print()

respuesta_segundo_mensaje = procesar_mensaje(
    numero_prueba,
    mensaje_prueba_2
)


# ------------------------------------------------------------
# 2. MOSTRAR RESPUESTA FINAL
# ------------------------------------------------------------

print()
print("RESPUESTA FINAL")
print("----------------------------------------")
print(respuesta_segundo_mensaje)


# ------------------------------------------------------------
# 3. VERIFICAR CANTIDAD DE MENSAJES EN MEMORIA
# ------------------------------------------------------------

print()
print("Mensajes almacenados en memoria:")
print(
    len(
        conversaciones_por_numero[
            numero_prueba
        ]["historial"]
    )
)

PROCESANDO SEGUNDO MENSAJE
----------------------------------------
Numero: 529331196350
Mensaje: Tiene 3 años y es un labrador. ¿Que necesito llevar?

Respuesta generada:
Con esa información, puedo asesorarte sobre las vacunas necesarias.

Para la cita, tienes que llevar el documento de identificación de tu perro (si lo tienes), las recetas de las vacunas anteriores (si las tiene) y la cartera de vacunas actualizada (si la tienes).

También te recomiendo llevar a tu perro con su collare y correa para que sea más fácil manejarlo durante la cita.

La cita es el jueves a las 18:00, ¿estás listo para esa fecha y hora? ¿Tienes alguna otra pregunta antes de la cita?
Respuesta enviada correctamente.
Destinatario: 529331196350
Respuesta generada:
Para la cita de vacunación de tu labrador, necesitarás llevar su documento de identificación con foto (si lo tienes) y llevarlo consigo con su correa favorita conmigo así será más fácil sacar las muestras que son necesarias para las vacunas.
Respuest

In [27]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - PRUEBA LIMPIA DEL LIMITE DE HISTORIAL
# CELDA 4 - PREPARAR UNA CONVERSACION NUEVA Y AISLADA
# ============================================================

# ------------------------------------------------------------
# 1. DEFINIR IDENTIFICADOR EXCLUSIVO PARA LA PRUEBA
# ------------------------------------------------------------

numero_prueba_memoria = "5210000000000"


# ------------------------------------------------------------
# 2. ELIMINAR CUALQUIER PRUEBA ANTERIOR CON ESTE IDENTIFICADOR
# ------------------------------------------------------------

if numero_prueba_memoria in conversaciones_por_numero:
    del conversaciones_por_numero[numero_prueba_memoria]


# ------------------------------------------------------------
# 3. CREAR UNA CONVERSACION COMPLETAMENTE NUEVA
# ------------------------------------------------------------

memoria_prueba = obtener_conversacion(
    numero_prueba_memoria
)


# ------------------------------------------------------------
# 4. VERIFICAR ESTADO INICIAL
# ------------------------------------------------------------

print()
print("PRUEBA LIMPIA DE MEMORIA")
print("----------------------------------------")
print("Identificador:", numero_prueba_memoria)
print("Conversation ID:", memoria_prueba["conversation_id"])
print(
    "Limite configurado:",
    MAX_MENSAJES_HISTORIAL,
    "mensajes"
)
print(
    "Mensajes almacenados inicialmente:",
    len(memoria_prueba["historial"])
)

assert len(memoria_prueba["historial"]) == 0

print()
print("Conversacion limpia preparada correctamente.")

Nueva conversacion creada
Numero: 5210000000000
Conversation ID: conv_a6fbcf59e7444b226378c502e7d6f594b2a71fe165fa9f92

PRUEBA LIMPIA DE MEMORIA
----------------------------------------
Identificador: 5210000000000
Conversation ID: conv_a6fbcf59e7444b226378c502e7d6f594b2a71fe165fa9f92
Limite configurado: 10 mensajes
Mensajes almacenados inicialmente: 0

Conversacion limpia preparada correctamente.


In [28]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - PRUEBA LIMPIA DEL LIMITE DE HISTORIAL
# CELDA 5 - CONVERSACION CONTROLADA DE CUATRO MENSAJES
# ============================================================

# ------------------------------------------------------------
# 1. DEFINIR LOS CUATRO MENSAJES DE LA CONVERSACION
# ------------------------------------------------------------

mensajes_prueba_memoria = [
    "Quiero una cita para vacunar a mi perro.",
    "Tiene 3 años y es un labrador.",
    "¿Que documentos necesito llevar?",
    "¿Debo llevar tambien su cartilla de vacunacion?"
]


# ------------------------------------------------------------
# 2. PROCESAR LOS CUATRO MENSAJES CON MEMORIA
# ------------------------------------------------------------

for numero_mensaje, texto in enumerate(
    mensajes_prueba_memoria,
    start=1
):

    print()
    print("MENSAJE", numero_mensaje)
    print("----------------------------------------")
    print("Usuario:", texto)

    respuesta = generar_respuesta_con_memoria(
        numero_prueba_memoria,
        texto
    )

    print("Asistente:", respuesta)

    cantidad = len(
        conversaciones_por_numero[
            numero_prueba_memoria
        ]["historial"]
    )

    print("Mensajes almacenados:", cantidad)


# ------------------------------------------------------------
# 3. MOSTRAR ESTADO DESPUES DE CUATRO INTERACCIONES
# ------------------------------------------------------------

historial_prueba = conversaciones_por_numero[
    numero_prueba_memoria
]["historial"]

print()
print("RESULTADO DESPUES DE CUATRO MENSAJES")
print("----------------------------------------")
print(
    "Limite configurado:",
    MAX_MENSAJES_HISTORIAL
)
print(
    "Mensajes almacenados:",
    len(historial_prueba)
)

assert len(historial_prueba) == 8

print()
print("Primera etapa de la prueba completada correctamente.")


MENSAJE 1
----------------------------------------
Usuario: Quiero una cita para vacunar a mi perro.
Asistente: Por favor, proporcioneme la información de tu mascota y a qué hora te gustaría programar la cita para la vacunación. ¿Cuál es el nombre de tu perro y cuál es su fecha de nacimiento? ¿Qué tipo de vacunas necesita?
Mensajes almacenados: 2

MENSAJE 2
----------------------------------------
Usuario: Tiene 3 años y es un labrador.
Asistente: ¡Excelente! Considerando que tu perro tiene 3 años, es probable que necesite la vacuna de rabia y la vacuna de parvovirus, entre otras. ¿Cuál es tu disponibilidad para la cita? Tenemos horarios disponibles mañana e incluso en este mismo día. ¿Qué día y a qué hora prefieres venir con tu perro a la consulta?
Mensajes almacenados: 4

MENSAJE 3
----------------------------------------
Usuario: ¿Que documentos necesito llevar?
Asistente: Para la primera consulta, necesitaremos ver los documentos de veterinaria de tu perro, como la tarjeta de iden

In [29]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - PRUEBA LIMPIA DEL LIMITE DE HISTORIAL
# CELDA 6 - SUPERAR EL LIMITE Y COMPROBAR EL RECORTE
# ============================================================

# ------------------------------------------------------------
# 1. DEFINIR DOS MENSAJES ADICIONALES
# ------------------------------------------------------------

mensajes_adicionales = [
    "¿Puedo darle de comer antes de la cita?",
    "¿Cuanto tiempo antes debo llegar a la veterinaria?"
]


# ------------------------------------------------------------
# 2. PROCESAR LOS MENSAJES 5 Y 6
# ------------------------------------------------------------

for numero_mensaje, texto in enumerate(
    mensajes_adicionales,
    start=5
):

    print()
    print("MENSAJE", numero_mensaje)
    print("----------------------------------------")
    print("Usuario:", texto)

    respuesta = generar_respuesta_con_memoria(
        numero_prueba_memoria,
        texto
    )

    print("Asistente:", respuesta)

    cantidad = len(
        conversaciones_por_numero[
            numero_prueba_memoria
        ]["historial"]
    )

    print("Mensajes almacenados:", cantidad)


# ------------------------------------------------------------
# 3. VERIFICAR EL RECORTE DEL HISTORIAL
# ------------------------------------------------------------

historial_final = conversaciones_por_numero[
    numero_prueba_memoria
]["historial"]

print()
print("VERIFICACION DEL LIMITE DE HISTORIAL")
print("----------------------------------------")
print("Limite configurado:", MAX_MENSAJES_HISTORIAL)
print("Mensajes almacenados finalmente:", len(historial_final))


# ------------------------------------------------------------
# 4. MOSTRAR HISTORIAL CONSERVADO
# ------------------------------------------------------------

print()
print("HISTORIAL CONSERVADO")
print("----------------------------------------")

for indice, mensaje in enumerate(
    historial_final,
    start=1
):
    print(
        indice,
        "|",
        mensaje["role"],
        "|",
        mensaje["content"]
    )


# ------------------------------------------------------------
# 5. VALIDACION AUTOMATICA
# ------------------------------------------------------------

assert len(historial_final) == MAX_MENSAJES_HISTORIAL

print()
print("Recorte del historial verificado correctamente.")
print(
    "El agente conserva solamente los",
    MAX_MENSAJES_HISTORIAL,
    "mensajes mas recientes."
)


MENSAJE 5
----------------------------------------
Usuario: ¿Puedo darle de comer antes de la cita?
Asistente: Sí, por supuesto. Puedes darle de comer antes de la cita, pero asegúrate de que tu perro vaya vacío estomacalmente al menos una hora antes. Si tu perro ha tenido un desayuno rico, no te preocupes, ya que estará bien sin problemas.
Mensajes almacenados: 10

MENSAJE 6
----------------------------------------
Usuario: ¿Cuanto tiempo antes debo llegar a la veterinaria?
Asistente: Te recomendamos llegar al menos 15 a 20 minutos antes de la cita programada. Esto nos permitirá revisar los documentos y prepararnos junto con tu perro. Si llegas un poco más temprano, estupendo, ya que podrás relajarte y pasar un rato en nuestro lobby mientras esperas. ¡Veremos pronto!
Mensajes almacenados: 10

VERIFICACION DEL LIMITE DE HISTORIAL
----------------------------------------
Limite configurado: 10
Mensajes almacenados finalmente: 10

HISTORIAL CONSERVADO
------------------------------------

In [30]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - EXPONER EL SERVIDOR Y PROBAR EL FLUJO COMPLETO
# CELDA 7 - VERIFICAR RESPUESTAS
# ============================================================

# ------------------------------------------------------------
# 1. RECUPERAR EL HISTORIAL FINAL DE LA PRUEBA
# ------------------------------------------------------------

historial_verificacion = conversaciones_por_numero[
    numero_prueba_memoria
]["historial"]


# ------------------------------------------------------------
# 2. VERIFICAR EL LIMITE CONFIGURADO
# ------------------------------------------------------------

print("VERIFICACION DE RESPUESTAS")
print("----------------------------------------")

print(
    "Limite configurado:",
    MAX_MENSAJES_HISTORIAL
)

print(
    "Mensajes conservados:",
    len(historial_verificacion)
)


# ------------------------------------------------------------
# 3. MOSTRAR LAS RESPUESTAS CONSERVADAS
# ------------------------------------------------------------

print()
print("RESPUESTAS DEL ASISTENTE")
print("----------------------------------------")

contador = 1

for mensaje in historial_verificacion:

    if mensaje["role"] == "assistant":

        print()
        print("Respuesta", contador)
        print(mensaje["content"])

        contador += 1


# ------------------------------------------------------------
# 4. VALIDAR EL RECORTE DEL HISTORIAL
# ------------------------------------------------------------

assert (
    len(historial_verificacion)
    <= MAX_MENSAJES_HISTORIAL
)

print()
print("----------------------------------------")
print("Respuestas verificadas correctamente.")
print("El contexto conversacional fue conservado.")
print("El limite del historial fue respetado.")


VERIFICACION DE RESPUESTAS
----------------------------------------
Limite configurado: 10
Mensajes conservados: 10

RESPUESTAS DEL ASISTENTE
----------------------------------------

Respuesta 1
¡Excelente! Considerando que tu perro tiene 3 años, es probable que necesite la vacuna de rabia y la vacuna de parvovirus, entre otras. ¿Cuál es tu disponibilidad para la cita? Tenemos horarios disponibles mañana e incluso en este mismo día. ¿Qué día y a qué hora prefieres venir con tu perro a la consulta?

Respuesta 2
Para la primera consulta, necesitaremos ver los documentos de veterinaria de tu perro, como la tarjeta de identificación sanitaria actualizada y el historial de vacunación anterior si lo tienes. También es conveniente llevar un mapa del lugar donde vives para facilitar el acceso a la consulta. Si tu perro tiene alfileres o marcas de identificación previas, por favor, tráelos también para registraslos en su historial.

Respuesta 3
Sí, la cartilla de vacunación (también conocida c

In [31]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - EXPONER EL SERVIDOR Y PROBAR EL FLUJO COMPLETO
# CELDA 8 - VER MENSAJES RECIBIDOS
# ============================================================

# ------------------------------------------------------------
# 1. MOSTRAR CANTIDAD DE EVENTOS RECIBIDOS POR EL WEBHOOK
# ------------------------------------------------------------

print("MENSAJES RECIBIDOS POR EL WEBHOOK")
print("----------------------------------------")

print(
    "Total de eventos almacenados:",
    len(mensajes_recibidos)
)


# ------------------------------------------------------------
# 2. EXTRAER LOS MENSAJES DE TEXTO RECIBIDOS
# ------------------------------------------------------------

for indice, evento in enumerate(
    mensajes_recibidos,
    start=1
):

    try:
        value = evento["entry"][0]["changes"][0]["value"]
        messages = value.get("messages", [])

        if messages:

            mensaje = messages[0]

            numero = mensaje.get("from", "")
            tipo = mensaje.get("type", "")

            print()
            print("Evento", indice)
            print("Numero:", numero)
            print("Tipo:", tipo)

            if tipo == "text":
                texto = mensaje.get(
                    "text",
                    {}
                ).get(
                    "body",
                    ""
                )

                print("Texto:", texto)

    except (KeyError, IndexError, TypeError):
        print()
        print(
            "Evento",
            indice,
            "sin mensaje de texto procesable."
        )


# ------------------------------------------------------------
# 3. CONFIRMAR INSPECCION
# ------------------------------------------------------------

print()
print("----------------------------------------")
print("Inspeccion de mensajes recibidos completada.")


MENSAJES RECIBIDOS POR EL WEBHOOK
----------------------------------------
Total de eventos almacenados: 2

Evento 1
Numero: 529331196350
Tipo: text
Texto: Quiero una cita para vacunar a mi perro.

Evento 2
Numero: 529331196350
Tipo: text
Texto: Tiene 3 años y es un labrador. ¿Que necesito llevar?

----------------------------------------
Inspeccion de mensajes recibidos completada.


In [32]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - EXPONER EL SERVIDOR Y PROBAR EL FLUJO COMPLETO
# CELDA 9 - VERIFICAR ALMACENAMIENTO Y CONVERSACION
# ============================================================

# ------------------------------------------------------------
# 1. RECUPERAR LA CONVERSACION DE LA PRUEBA LIMPIA
# ------------------------------------------------------------

memoria_verificacion = conversaciones_por_numero[
    numero_prueba_memoria
]

conversation_id = memoria_verificacion[
    "conversation_id"
]

historial_verificacion = memoria_verificacion[
    "historial"
]


# ------------------------------------------------------------
# 2. MOSTRAR IDENTIFICADOR CREADO EN LLAMA STACK
# ------------------------------------------------------------

print("VERIFICACION DE CONVERSACION")
print("----------------------------------------")

print("Numero de prueba:")
print(numero_prueba_memoria)

print()
print("Conversation ID creado en Llama Stack:")
print(conversation_id)


# ------------------------------------------------------------
# 3. VERIFICAR EL HISTORIAL ADMINISTRADO POR EL AGENTE
# ------------------------------------------------------------

print()
print("Historial administrado por el agente:")
print(len(historial_verificacion), "mensajes")

print(
    "Limite configurado:",
    MAX_MENSAJES_HISTORIAL,
    "mensajes"
)


# ------------------------------------------------------------
# 4. MOSTRAR HISTORIAL CONSERVADO
# ------------------------------------------------------------

print()
print("HISTORIAL CONSERVADO")
print("----------------------------------------")

for indice, mensaje in enumerate(
    historial_verificacion,
    start=1
):

    print(
        indice,
        "|",
        mensaje["role"],
        "|",
        mensaje["content"]
    )


# ------------------------------------------------------------
# 5. VALIDAR CONVERSACION Y LIMITE
# ------------------------------------------------------------

assert conversation_id
assert len(historial_verificacion) <= MAX_MENSAJES_HISTORIAL

print()
print("----------------------------------------")
print("Conversation ID verificado correctamente.")
print("Historial del agente verificado correctamente.")
print("Limite de memoria respetado correctamente.")


VERIFICACION DE CONVERSACION
----------------------------------------
Numero de prueba:
5210000000000

Conversation ID creado en Llama Stack:
conv_a6fbcf59e7444b226378c502e7d6f594b2a71fe165fa9f92

Historial administrado por el agente:
10 mensajes
Limite configurado: 10 mensajes

HISTORIAL CONSERVADO
----------------------------------------
1 | user | Tiene 3 años y es un labrador.
2 | assistant | ¡Excelente! Considerando que tu perro tiene 3 años, es probable que necesite la vacuna de rabia y la vacuna de parvovirus, entre otras. ¿Cuál es tu disponibilidad para la cita? Tenemos horarios disponibles mañana e incluso en este mismo día. ¿Qué día y a qué hora prefieres venir con tu perro a la consulta?
3 | user | ¿Que documentos necesito llevar?
4 | assistant | Para la primera consulta, necesitaremos ver los documentos de veterinaria de tu perro, como la tarjeta de identificación sanitaria actualizada y el historial de vacunación anterior si lo tienes. También es conveniente llevar un mapa

In [33]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - PRUEBA DE REGLA FIJA DE URGENCIA
# CELDA 10 - VERIFICAR DETECCION DE URGENCIA VETERINARIA
# ============================================================

# ------------------------------------------------------------
# 1. DEFINIR MENSAJE DE URGENCIA
# ------------------------------------------------------------

mensaje_urgencia = (
    "Mi perro tiene un sangrado y necesito ayuda."
)


# ------------------------------------------------------------
# 2. COMPROBAR LA REGLA FIJA
# ------------------------------------------------------------

print("PRUEBA DE URGENCIA VETERINARIA")
print("----------------------------------------")
print("Mensaje:", mensaje_urgencia)

urgencia_detectada = es_urgencia_veterinaria(
    mensaje_urgencia
)

print()
print("Urgencia detectada:", urgencia_detectada)


# ------------------------------------------------------------
# 3. GENERAR RESPUESTA FIJA SIN LLAMAR A LLAMA
# ------------------------------------------------------------

if urgencia_detectada:

    respuesta_urgencia = (
        "Tu mensaje indica una posible urgencia veterinaria. "
        "Busca atencion veterinaria inmediata y comunicate "
        "directamente con la clinica o servicio de emergencias."
    )

    print()
    print("REGLA FIJA ACTIVADA")
    print("----------------------------------------")
    print(respuesta_urgencia)

else:

    raise AssertionError(
        "La regla de urgencia no fue activada."
    )


# ------------------------------------------------------------
# 4. VALIDACION FINAL
# ------------------------------------------------------------

assert urgencia_detectada is True

print()
print("----------------------------------------")
print("Prueba de urgencia completada correctamente.")
print("La respuesta fija fue utilizada.")
print("Llama no fue consultado para generar esta respuesta.")

PRUEBA DE URGENCIA VETERINARIA
----------------------------------------
Mensaje: Mi perro tiene un sangrado y necesito ayuda.

Urgencia detectada: True

REGLA FIJA ACTIVADA
----------------------------------------
Tu mensaje indica una posible urgencia veterinaria. Busca atencion veterinaria inmediata y comunicate directamente con la clinica o servicio de emergencias.

----------------------------------------
Prueba de urgencia completada correctamente.
La respuesta fija fue utilizada.
Llama no fue consultado para generar esta respuesta.


In [34]:
# ============================================================
# CHALLENGE: AGENTE DE UNA VETERINARIA
# PUNTO 5 - EXPONER EL SERVIDOR Y PROBAR EL FLUJO COMPLETO
# CELDA 11 - FINALIZAR PROCESO DE TUNELIZACION
# ============================================================

from pyngrok import ngrok

# ------------------------------------------------------------
# 1. MOSTRAR ESTADO DEL CIERRE
# ------------------------------------------------------------

print("FINALIZANDO TUNEL NGROK")
print("----------------------------------------")


# ------------------------------------------------------------
# 2. CERRAR LOS TUNELES ACTIVOS
# ------------------------------------------------------------

try:
    ngrok.kill()

    print("Tunel ngrok finalizado correctamente.")

except Exception as error:

    print("Se produjo un aviso durante el cierre:")
    print(error)


# ------------------------------------------------------------
# 3. CONFIRMAR FINALIZACION
# ------------------------------------------------------------

print()
print("----------------------------------------")
print("Proceso de tunelizacion finalizado.")
print("Challenge del agente de veterinaria completado.")


FINALIZANDO TUNEL NGROK
----------------------------------------
Tunel ngrok finalizado correctamente.

----------------------------------------
Proceso de tunelizacion finalizado.
Challenge del agente de veterinaria completado.
